<a href="https://colab.research.google.com/github/AndikaPutra509/Prediksi-Saham/blob/main/Trading_Scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
"""
📈 IDX STOCK SCANNER - QUADRUPLE ENGINE (FULL SYSTEM)
Fitur:
- Phase 1: Trading Scanner dengan Modal-Adaptive Filters (Data Warehouse)
- Phase 2: Walk-Forward Validation, Sensitivity Analysis
- Phase 2.3: Monte Carlo FIXED (realistis)
- Export: Google Sheets (LANGSUNG KE DRIVE, TANPA FILE)
- Panduan Portfolio: 3 rekomendasi terbaik dengan alokasi modal

Author: Quant System
Date: 2026
"""

# =============================================================================
# 1. INSTALL DEPENDENCIES & IMPORTS
# =============================================================================

from google.colab import auth
from google.auth import default
import gspread
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
from datetime import datetime, timedelta
import time
import pickle
import os
from tabulate import tabulate
from collections import defaultdict
import logging
import random
from typing import Optional, Dict, List, Tuple, Union, Any
import hashlib
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Matikan logging yang tidak perlu
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

print("✅ Dependencies imported")

# =============================================================================
# 2. STOCK UNIVERSE (FULL)
# =============================================================================

STOCKBIT_UNIVERSE = [
    "AALI", "ABBA", "ABDA", "ABMM", "ACES", "ADES", "ADHI", "ADMF", "ADMG", "ADRO",
    "AGAR", "AGII", "AGRO", "AHAP", "AIMS", "AISA", "AKRA", "AKSI", "ALDO", "ALKA",
    "ALMI", "AMAG", "AMFG", "AMMN", "AMRT", "ANDI", "ANJT", "ANTM", "APEX", "APIC",
    "APLN", "ARCI", "ARGO", "ARII", "ARNA", "ARTA", "ARTO", "ASBI", "ASDM", "ASGR",
    "ASHA", "ASII", "ASLI", "ASMI", "ASPI", "ASRI", "ASRM", "ASSA", "ATLA", "AUTO",
    "AVIA", "AWAN", "AYLS", "BABP", "BACA", "BALI", "BANK", "BAPA", "BAPI", "BATA",
    "BAYU", "BBCA", "BBHI", "BBKP", "BBLD", "BBNI", "BBRI", "BBRM", "BBSS", "BBTN",
    "BBYB", "BCAP", "BCIC", "BDMN", "BEEF", "BEKS", "BELL", "BEST", "BFIN", "BGTG",
    "BHAT", "BIMA", "BINA", "BIPP", "BIPI", "BIRD", "BISI", "BJBR", "BJTM", "BKSL",
    "BLTA", "BLUE", "BMAS", "BMBL", "BMRI", "BMSR", "BMTR", "BNBA", "BNBR", "BNGA",
    "BNII", "BNLI", "BOBA", "BOLT", "BOSS", "BPFI", "BPII", "BPTR", "BRAM", "BREN",
    "BRIS", "BRMS", "BRNA", "BRPT", "BSDE", "BSIM", "BSSR", "BTEL", "BTON", "BTPN",
    "BTPS", "BUDI", "BULL", "BUMI", "BUVA", "BWPT", "BYAN", "CAMP", "CANI", "CARS",
    "CASA", "CASS", "CBDK", "CBMF", "CCSI", "CDAX", "CEKA", "CENT", "CFIN", "CITA",
    "CITY", "CKRA", "CLEO", "CLPI", "CMNP", "CMPP", "CMRY", "CNKO", "CNTX", "COAL",
    "COCO", "COWL", "CPIN", "CPRO", "CSAP", "CSIS", "CTBN", "CTRA", "CTTH", "CUAN",
    "DAAZ", "DART", "DASA", "DAYA", "DCII", "DEGA", "DEWA", "DFAM", "DGIK", "DIGI",
    "DILD", "DIVA", "DIVN", "DKFT", "DLTA", "DMAS", "DMND", "DNAR", "DNET", "DNLS",
    "DOID", "DOOH", "DPNS", "DPUM", "DSFI", "DSNG", "DSSA", "DUCK", "DUTI", "DVLA",
    "DYAN", "EASI", "EASY", "EBMT", "ECII", "EDGE", "EKAD", "ELBA", "ELSA", "ELTY",
    "EMBR", "EMDE", "EMTK", "ENRG", "ENVY", "ENZO", "EPAC", "EPMT", "ERAA", "ERTX",
    "ESSA", "ESTA", "ESTI", "ETWA", "EXCL", "FAST", "FASW", "FILM", "FISH", "FITT",
    "FKSF", "FLMC", "FMII", "FORE", "FORU", "FORZ", "FPNI", "FREN", "FUJI", "FUTR",
    "GAMA", "GDST", "GDYR", "GEMS", "GGRM", "GGRP", "GHON", "GIDS", "GJTL", "GLVA",
    "GMFI", "GMTD", "GOLD", "GOOD", "GOTO", "GPRA", "GRPH", "GSMF", "GTBO", "GTRA",
    "GTSI", "GULA", "GZCO", "HADE", "HDFA", "HDIT", "HEAL", "HERO", "HITS", "HKMU",
    "HMSP", "HOKI", "HOMI", "HOPE", "HOTL", "HRME", "HRTA", "HRUM", "HSBK", "HSMP",
    "HUMI", "IBFN", "IBOS", "IBST", "ICBP", "ICON", "IDPR", "IFII", "IFSH", "IGAR",
    "IIKP", "IKAI", "IKAN", "IMAS", "IMJS", "IMPC", "INAF", "INAI", "INCF", "INCI",
    "INCO", "INDF", "INDS", "INDX", "INDY", "INET", "INKP", "INPC", "INPP", "INPS",
    "INRU", "INTA", "INTD", "INTP", "IPCC", "IPCM", "IPOL", "IRRA", "ISAT", "ISEA",
    "ISSP", "ITIC", "ITMG", "JAST", "JAWA", "JAYA", "JECC", "JEMB", "JFAS", "JGLE",
    "JHON", "JIHD", "JKON", "JKSW", "JMAS", "JPFA", "JPII", "JPUR", "JRPT", "JSKY",
    "JSMR", "JSPT", "JTNB", "KAEF", "KAQI", "KARW", "KBLI", "KBLM", "KBRT", "KBRI",
    "KDSI", "KDTN", "KEEN", "KETR", "KICI", "KIJA", "KINO", "KIOS", "KJEN", "KKGI",
    "KLBF", "KMTR", "KOBX", "KOIN", "KOLI", "KONI", "KOTA", "KPAL", "KPIG", "KRAS",
    "KREN", "KRYA", "KSEL", "KUAS", "KUIC", "KUVO", "LAND", "LAPD", "LATA", "LBAK",
    "LCGP", "LCKM", "LEAD", "LIFE", "LINK", "LION", "LISA", "LMAS", "LMPI", "LMSH",
    "LPCK", "LPGI", "LPIN", "LPKR", "LPLI", "LPPF", "LPPS", "LSIP", "LSPI", "LTLS",
    "LUCY", "MABA", "MABH", "MAGP", "MAIN", "MAMI", "MAPA", "MAPB", "MAPI", "MARA",
    "MASA", "MAYA", "MBAP", "MBCA", "MBMA", "MBSS", "MBTO", "MCAS", "MCPI", "MCOR",
    "MDIA", "MDKA", "MDKI", "MEDC", "MEDS", "MEGA", "MERK", "META", "MFIN", "MFMI",
    "MGLV", "MGNA", "MGRO", "MIDI", "MIKA", "MINA", "MIRA", "MITI", "MITT", "MKNT",
    "MKPI", "MLBI", "MLIA", "MLPL", "MLPT", "MLSL", "MMIX", "MMLP", "MNCN", "MOLI",
    "MPOW", "MPPA", "MPRO", "MPTJ", "MRAT", "MSIE", "MSIN", "MSKY", "MTDL", "MTFN",
    "MTLA", "MTPS", "MTSM", "MUDA", "MUTU", "MYOH", "MYOR", "MYRX", "MYSX", "NAGA",
    "NASI", "NATO", "NAYZ", "NCKL", "NELY", "NETV", "NFCX", "NICL", "NIKL", "NISP",
    "NITY", "NIYM", "NOBU", "NPGF", "NRCA", "NSSS", "NTBK", "NUSA", "NUSI", "OASA",
    "OCTN", "OKAS", "OMED", "ONIX", "OPMS", "ORNA", "OTBK", "PADA", "PADI", "PAMG",
    "PANR", "PANS", "PANU", "PAPA", "PASA", "PASS", "PBRX", "PBID", "PBSA", "PCAR",
    "PDES", "PDGD", "PDIN", "PEGE", "PGAS", "PGLI", "PGUN", "PICO", "PIDRA", "PJAA",
    "PKPK", "PLAN", "PLAS", "PLIN", "PMJS", "PMMP", "PNBN", "PNBS", "PNIN", "PNLF",
    "PNSE", "POLI", "POLL", "POLU", "POLY", "POOL", "PORT", "POWR", "PPGL", "PPRE",
    "PPRO", "PPSI", "PRAS", "PRDA", "PRIM", "PRIN", "PRLD", "PROD", "PROT", "PRTS",
    "PSAB", "PSBA", "PSDN", "PSGO", "PSKT", "PSSI", "PTBA", "PTDU", "PTIS", "PTMP",
    "PTPP", "PTPW", "PTRO", "PTSN", "PTSP", "PUDP", "PURA", "PURE", "PWON", "PYFA",
    "RACE", "RADIO", "RAFI", "RAJA", "RAKD", "RALS", "RANC", "RATU", "RBMS", "RDTX",
    "REAL", "RELI", "RIGS", "RIMO", "RISE", "RMBA", "RMKE", "ROCK", "RODA", "ROKI",
    "ROTI", "RRMI", "RUIS", "RUMI", "SABA", "SAFE", "SAME", "SAPX", "SARA", "SATO",
    "SBAT", "SBBP", "SBGA", "SBMA", "SBMF", "SCBD", "SCCC", "SCCO", "SCMA", "SCNP",
    "SDPC", "SDRA", "SEAN", "SECR", "SEMA", "SFAN", "SGER", "SGRO", "SHID", "SHIP",
    "SIDO", "SILO", "SIMA", "SIMP", "SIPD", "SIPO", "SKBM", "SKLT", "SKRN", "SLIS",
    "SMAR", "SMDR", "SMGR", "SMIL", "SMMT", "SMSM", "SMRA", "SNLK", "SNMS", "SOFA",
    "SONA", "SOSS", "SOUL", "SPMA", "SPMI", "SPNA", "SPRE", "SPTO", "SQBI", "SQMI",
    "SRAJ", "SRIL", "SRSN", "SSIA", "SSMS", "SSTM", "STAR", "STTP", "SUGI", "SULI",
    "SUPR", "SURI", "SWAT", "SWID", "TALD", "TAMA", "TAMU", "TAPG", "TARA", "TASP",
    "TATA", "TAXI", "TBIG", "TBLA", "TCID", "TDPM", "TELE", "TEMB", "TEMPO", "TIFA",
    "TIGA", "TINS", "TIRA", "TIRT", "TITA", "TKGA", "TKIM", "TLKM", "TMAS", "TMPO",
    "TMSH", "TOBA", "TOOL", "TOPS", "TOSK", "TOTL", "TOTO", "TOWR", "TPIA", "TPMA",
    "TRAM", "TRGU", "TRIO", "TRIS", "TRJA", "TRON", "TRST", "TRUB", "TRUK", "TRUS",
    "TSPC", "TUGU", "TURI", "TUVN", "TYRE", "UANG", "UCID", "UDIJ", "UFNX", "UGRO",
    "UJSN", "ULTJ", "UNIC", "UNIQ", "UNIT", "UNSP", "UNTR", "UNVR", "USFI", "VALU",
    "VICO", "VICI", "VIDI", "VISI", "VIVA", "VKTR", "VOKS", "VRNA", "VTNY", "WAPO",
    "WEGE", "WEHA", "WICO", "WIFI", "WIIM", "WINS", "WMUU", "WMPP", "WOOD", "WOWS",
    "WRKR", "WSBP", "WSKT", "WTON", "YELO", "YULE", "ZBRA", "ZINC", "ZONE"
]

# =============================================================================
# 3. QUALITY STOCKS UNIVERSE (IDX30 + LQ45 + IDX80)
# =============================================================================

QUALITY_STOCKS = [
    'ADRO', 'AMRT', 'ANTM', 'ASII', 'BBCA', 'BBNI', 'BBRI', 'BMRI', 'BREN', 'BUMI',
    'CPIN', 'EMTK', 'GOTO', 'ICBP', 'INCO', 'INDF', 'INKP', 'ISAT', 'JPFA', 'JSMR',
    'KLBF', 'MDKA', 'MEDC', 'MIKA', 'MTEL', 'NCKL', 'PGAS', 'TLKM', 'TOWR', 'UNTR',
    'AKRA', 'ARTO', 'BBTN', 'BDKR', 'BIRD', 'BJBR', 'BJTM', 'BRIS', 'BRPT', 'BUKA',
    'EXCL', 'HRUM', 'INTP', 'ITMG', 'MPMX', 'PTBA', 'SIDO', 'SMGR', 'TPIA', 'UNVR',
    'WIKA', 'ACES', 'ADHI', 'AGRO', 'AALI', 'ARNA', 'ASGR', 'ASRI', 'AUTO', 'BAYU',
    'BEST', 'BFIN', 'BIPP', 'BISI', 'BKSL', 'BLTA', 'BMAS', 'BMSR', 'BNGA', 'BNII',
    'BOBA', 'BOLT', 'BOSS', 'BPFI', 'BRAM', 'BSDE', 'BSSR', 'BTON', 'BUDI', 'BULL',
    'BUVA', 'BWPT', 'BYAN', 'CAMP', 'CANI', 'CARS', 'CASA', 'CASS', 'CBDK', 'CBMF',
    'CCSI', 'CDAX', 'CEKA', 'CENT', 'CFIN', 'CITA', 'CITY', 'CKRA', 'CLEO', 'CLPI',
    'CMNP', 'CMPP', 'CMRY', 'CNKO', 'COAL', 'COCO', 'COWL', 'CPRO', 'CSAP', 'CSIS',
    'CTBN', 'CTRA', 'CTTH', 'CUAN', 'DART', 'DASA', 'DCII', 'DEGA', 'DEWA', 'DGIK',
    'DIGI', 'DILD', 'DIVA', 'DIVN', 'DLTA', 'DMAS', 'DMND', 'DNAR', 'DNET', 'DNLS',
    'DOID', 'DOOH', 'DPNS', 'DSFI', 'DSNG', 'DSSA', 'DUCK', 'DUTI', 'DVLA', 'DYAN',
    'EASI', 'EASY', 'EBMT', 'ECII', 'EDGE', 'EKAD', 'ELBA', 'ELSA', 'ELTY', 'EMBR',
    'EMDE', 'ENRG', 'ENVY', 'ENZO', 'EPAC', 'EPMT', 'ERAA', 'ESSA', 'ESTA', 'ESTI',
    'ETWA', 'FAST', 'FASW', 'FILM', 'FISH', 'FITT', 'FKSF', 'FLMC', 'FMII', 'FORE',
    'FORU', 'FORZ', 'FPNI', 'FREN', 'FUJI', 'FUTR', 'GAMA', 'GDST', 'GDYR', 'GEMS',
    'GGRM', 'GGRP', 'GHON', 'GIDS', 'GJTL', 'GLVA', 'GMFI', 'GMTD', 'GOLD', 'GOOD',
    'GPRA', 'GRPH', 'GSMF', 'GTBO', 'GTRA', 'GTSI', 'GULA', 'HADE', 'HDFA', 'HDIT',
    'HEAL', 'HERO', 'HITS', 'HKMU', 'HMSP', 'HOKI', 'HOMI', 'HOPE', 'HOTL', 'HRME',
    'HRTA', 'HSBK', 'HSMP', 'HUMI', 'IBFN', 'IBOS', 'IBST', 'ICON', 'IDPR', 'IFII',
    'IFSH', 'IGAR', 'IIKP', 'IKAI', 'IKAN', 'IMAS', 'IMJS', 'IMPC', 'INAF', 'INAI',
    'INCF', 'INCI', 'INDS', 'INDX', 'INDY', 'INET', 'INKP', 'INPC', 'INPP', 'INPS',
    'INRU', 'INTA', 'INTD', 'INTP', 'IPCC', 'IPCM', 'IPOL', 'IRRA', 'ISEA', 'ISSP',
    'ITIC', 'JAST', 'JAWA', 'JAYA', 'JECC', 'JEMB', 'JFAS', 'JGLE', 'JHON', 'JIHD',
    'JKON', 'JKSW', 'JMAS', 'JPII', 'JPUR', 'JRPT', 'JSKY', 'JSPT', 'JTNB', 'KAEF',
    'KAQI', 'KARW', 'KBLI', 'KBLM', 'KBRT', 'KBRI', 'KDSI', 'KDTN', 'KEEN', 'KETR',
    'KICI', 'KIJA', 'KINO', 'KIOS', 'KJEN', 'KKGI', 'KMTR', 'KOBX', 'KOIN', 'KOLI',
    'KONI', 'KOTA', 'KPAL', 'KPIG', 'KRAS', 'KREN', 'KRYA', 'KSEL', 'KUAS', 'KUIC',
    'KUVO', 'LAND', 'LAPD', 'LATA', 'LBAK', 'LCGP', 'LCKM', 'LEAD', 'LIFE', 'LINK',
    'LION', 'LISA', 'LMAS', 'LMPI', 'LMSH', 'LPCK', 'LPGI', 'LPIN', 'LPKR', 'LPLI',
    'LPPF', 'LPPS', 'LSIP', 'LSPI', 'LTLS', 'LUCY', 'MABA', 'MABH', 'MAGP', 'MAIN',
    'MAMI', 'MAPA', 'MAPB', 'MAPI', 'MARA', 'MASA', 'MAYA', 'MBAP', 'MBCA', 'MBMA',
    'MBSS', 'MBTO', 'MCAS', 'MCPI', 'MCOR', 'MDIA', 'MDKA', 'MDKI', 'MEDC', 'MEDS',
    'MEGA', 'MERK', 'META', 'MFIN', 'MFMI', 'MGLV', 'MGNA', 'MGRO', 'MIDI', 'MIKA',
    'MINA', 'MIRA', 'MITI', 'MITT', 'MKNT', 'MKPI', 'MLBI', 'MLIA', 'MLPL', 'MLPT',
    'MLSL', 'MMIX', 'MMLP', 'MNCN', 'MOLI', 'MPOW', 'MPPA', 'MPRO', 'MPTJ', 'MRAT',
    'MSIE', 'MSIN', 'MSKY', 'MTDL', 'MTFN', 'MTLA', 'MTPS', 'MTSM', 'MUDA', 'MUTU',
    'MYOH', 'MYOR', 'MYRX', 'MYSX', 'NAGA', 'NASI', 'NATO', 'NAYZ', 'NCKL', 'NELY',
    'NETV', 'NFCX', 'NICL', 'NIKL', 'NISP', 'NITY', 'NIYM', 'NOBU', 'NPGF', 'NRCA',
    'NSSS', 'NTBK', 'NUSA', 'NUSI', 'OASA', 'OCTN', 'OKAS', 'OMED', 'ONIX', 'OPMS',
    'ORNA', 'OTBK', 'PADA', 'PADI', 'PAMG', 'PANR', 'PANS', 'PANU', 'PAPA', 'PASA',
    'PASS', 'PBRX', 'PBID', 'PBSA', 'PCAR', 'PDES', 'PDGD', 'PDIN', 'PEGE', 'PGAS',
    'PGLI', 'PGUN', 'PICO', 'PIDRA', 'PJAA', 'PKPK', 'PLAN', 'PLAS', 'PLIN', 'PMJS',
    'PMMP', 'PNBN', 'PNBS', 'PNIN', 'PNLF', 'PNSE', 'POLI', 'POLL', 'POLU', 'POLY',
    'POOL', 'PORT', 'POWR', 'PPGL', 'PPRE', 'PPRO', 'PPSI', 'PRAS', 'PRDA', 'PRIM',
    'PRIN', 'PRLD', 'PROD', 'PROT', 'PRTS', 'PSAB', 'PSBA', 'PSDN', 'PSGO', 'PSKT',
    'PSSI', 'PTDU', 'PTIS', 'PTMP', 'PTPP', 'PTPW', 'PTRO', 'PTSN', 'PTSP', 'PUDP',
    'PURA', 'PURE', 'PWON', 'PYFA', 'RACE', 'RADIO', 'RAFI', 'RAJA', 'RAKD', 'RALS',
    'RANC', 'RATU', 'RBMS', 'RDTX', 'REAL', 'RELI', 'RIGS', 'RIMO', 'RISE', 'RMBA',
    'RMKE', 'ROCK', 'RODA', 'ROKI', 'ROTI', 'RRMI', 'RUIS', 'RUMI', 'SABA', 'SAFE',
    'SAME', 'SAPX', 'SARA', 'SATO', 'SBAT', 'SBBP', 'SBGA', 'SBMA', 'SBMF', 'SCBD',
    'SCCC', 'SCCO', 'SCMA', 'SCNP', 'SDPC', 'SDRA', 'SEAN', 'SECR', 'SEMA', 'SFAN',
    'SGER', 'SGRO', 'SHID', 'SHIP', 'SIDO', 'SILO', 'SIMA', 'SIMP', 'SIPD', 'SIPO',
    'SKBM', 'SKLT', 'SKRN', 'SLIS', 'SMAR', 'SMDR', 'SMIL', 'SMMT', 'SMSM', 'SMRA',
    'SNLK', 'SNMS', 'SOFA', 'SONA', 'SOSS', 'SOUL', 'SPMA', 'SPMI', 'SPNA', 'SPRE',
    'SPTO', 'SQBI', 'SQMI', 'SRAJ', 'SRIL', 'SRSN', 'SSIA', 'SSMS', 'SSTM', 'STAR',
    'STTP', 'SUGI', 'SULI', 'SUPR', 'SURI', 'SWAT', 'SWID', 'TALD', 'TAMA', 'TAMU',
    'TAPG', 'TARA', 'TASP', 'TATA', 'TAXI', 'TBIG', 'TBLA', 'TCID', 'TDPM', 'TELE',
    'TEMB', 'TEMPO', 'TIFA', 'TIGA', 'TINS', 'TIRA', 'TIRT', 'TITA', 'TKGA', 'TKIM',
    'TLKM', 'TMAS', 'TMPO', 'TMSH', 'TOBA', 'TOOL', 'TOPS', 'TOSK', 'TOTL', 'TOTO',
    'TPMA', 'TRAM', 'TRGU', 'TRIO', 'TRIS', 'TRJA', 'TRON', 'TRST', 'TRUB', 'TRUK',
    'TRUS', 'TSPC', 'TUGU', 'TURI', 'TUVN', 'TYRE', 'UANG', 'UCID', 'UDIJ', 'UFNX',
    'UGRO', 'UJSN', 'ULTJ', 'UNIC', 'UNIQ', 'UNIT', 'UNSP', 'USFI', 'VALU', 'VICO',
    'VICI', 'VIDI', 'VISI', 'VIVA', 'VKTR', 'VOKS', 'VRNA', 'VTNY', 'WAPO', 'WEGE',
    'WEHA', 'WICO', 'WIFI', 'WIIM', 'WINS', 'WMUU', 'WMPP', 'WOOD', 'WOWS', 'WRKR',
    'WSBP', 'WSKT', 'WTON', 'YELO', 'YULE', 'ZBRA', 'ZINC', 'ZONE'
]
QUALITY_STOCKS = list(dict.fromkeys(QUALITY_STOCKS))

# =============================================================================
# 4. SECTOR MAPPING
# =============================================================================

ENERGY_SECTOR = ['ADRO', 'PTBA', 'ITMG', 'MEDC', 'PGAS', 'ENRG', 'BUMI', 'DOID']
MINING_GOLD = ['ANTM', 'MDKA', 'BRMS']
EXPORT_SECTOR = ['ANTM', 'INCO', 'TINS', 'HRUM', 'CPIN', 'JPFA', 'MAIN']

SECTOR_MAPPING = {
    'ENERGY': ENERGY_SECTOR + ['BYAN', 'INDY', 'HRUM', 'ARTI'],
    'MINING': MINING_GOLD + ['INCO', 'TINS', 'CITA', 'DKFT'],
    'FINANCE': ['BBCA', 'BBRI', 'BMRI', 'BBNI', 'PNBN', 'BJBR', 'BJTM', 'NISP', 'BDMN', 'BNLI', 'BNGA', 'BNII', 'BSIM'],
    'PROPERTY': ['PWON', 'BSDE', 'LPKR', 'CTRA', 'SMRA', 'ASRI', 'DILD', 'MDLN', 'ELTY', 'BIPP', 'BKSL', 'MTLA', 'MAPI'],
    'CONSUMER': ['UNVR', 'ICBP', 'INDF', 'GGRM', 'HMSP', 'KLBF', 'MYOR', 'SIDO', 'ULTJ', 'DLTA', 'MLBI', 'TCID', 'ROTI', 'SKBM'],
    'INFRASTRUCTURE': ['TLKM', 'JSMR', 'TOWR', 'TBIG', 'WIKA', 'PTPP', 'WSKT', 'ADHI', 'TOTL', 'ACST'],
    'INDUSTRIAL': ['ASII', 'GJTL', 'AUTO', 'BRAM', 'INDS', 'BOLT', 'IMP', 'PRAS', 'PBRX'],
    'TRADE': ['MAPI', 'ACES', 'RALS', 'LPPF', 'ERAA', 'MAPB', 'SONA', 'CSAP', 'MIDI', 'MFMI'],
    'AGRICULTURE': ['AALI', 'LSIP', 'SGRO', 'BWPT', 'SMAR', 'DSNG', 'JAWA'],
    'HEALTHCARE': ['MIKA', 'SILO', 'KLBF', 'KAEF', 'INAF', 'DVLA', 'TSPC', 'MERK', 'SCPI']
}

def get_sector(symbol: str) -> str:
    """Get sector for a symbol"""
    for sector, stocks in SECTOR_MAPPING.items():
        if symbol in stocks:
            return sector
    return 'OTHER'

# =============================================================================
# 5. GLOBAL INDICES CONFIGURATION
# =============================================================================

GLOBAL_INDICES = {
    "IHSG": {
        "ticker": "^JKSE",
        "nama": "Indeks Harga Saham Gabungan",
        "satuan": "poin",
        "keterangan": "Indeks utama BEI"
    },
    "DOWJONES": {
        "ticker": "^DJI",
        "nama": "Dow Jones Industrial Average",
        "satuan": "poin",
        "keterangan": "Indeks saham AS"
    },
    "USDIDR": {
        "ticker": "IDR=X",
        "nama": "USD/IDR",
        "satuan": "Rp/USD",
        "keterangan": "Nilai tukar Rupiah terhadap Dolar"
    },
    "OIL": {
        "ticker": "CL=F",
        "nama": "Crude Oil WTI",
        "satuan": "USD/barel",
        "keterangan": "Harga minyak dunia"
    },
    "GOLD": {
        "ticker": "GC=F",
        "nama": "Gold Futures",
        "satuan": "USD/ons",
        "keterangan": "Harga emas dunia"
    }
}

# =============================================================================
# 6. GLOBAL INDICES FETCHER
# =============================================================================

class GlobalIndicesFetcher:
    """Fetch global indices data for market context dengan transparansi penuh"""

    def __init__(self):
        self.data = {}
        self.momentum = {}
        self.status = {}
        self.prices = {}
        self.detail = {}

    def _to_scalar(self, value: Any) -> float:
        if value is None:
            return 0.0
        if isinstance(value, (np.ndarray, list)):
            if len(value) > 0:
                return float(value[0])
            return 0.0
        return float(value)

    def fetch_all(self) -> None:
        print("\n" + "="*80)
        print("📡 FETCHING GLOBAL INDICES - DETAIL TRANSPARAN")
        print("="*80)

        end_date = datetime.now()
        start_date = end_date - timedelta(days=365*2)

        for name, config in GLOBAL_INDICES.items():
            ticker = config["ticker"]
            print(f"\n📊 {name}: {config['nama']}")
            print(f"   Ticker: {ticker}")
            print(f"   Satuan: {config['satuan']}")
            print(f"   Keterangan: {config['keterangan']}")

            try:
                df = yf.download(
                    ticker,
                    start=start_date.strftime("%Y-%m-%d"),
                    end=end_date.strftime("%Y-%m-%d"),
                    interval="1d",
                    auto_adjust=True,
                    progress=False,
                    timeout=10
                )

                time.sleep(0.5)

                if df.empty or len(df) < 200:
                    self.status[name] = "UNAVAILABLE"
                    self.momentum[name] = 0.0
                    self.prices[name] = 0.0
                    print(f"   ⚠️  Status: TIDAK TERSEDIA (data tidak cukup)")
                else:
                    close = df['Close'].values
                    current_price = float(close[-1])
                    price_1w_ago = float(close[-6]) if len(close) >= 6 else current_price
                    price_1m_ago = float(close[-22]) if len(close) >= 22 else current_price
                    price_3m_ago = float(close[-66]) if len(close) >= 66 else current_price
                    price_1y_ago = float(close[-252]) if len(close) >= 252 else current_price

                    momentum_1w = (current_price / price_1w_ago - 1) * 100
                    momentum_1m = (current_price / price_1m_ago - 1) * 100
                    momentum_3m = (current_price / price_3m_ago - 1) * 100
                    momentum_1y = (current_price / price_1y_ago - 1) * 100 if len(close) >= 252 else 0

                    if name == "IHSG" and len(close) >= 200:
                        ma50 = np.mean(close[-50:])
                        ma200 = np.mean(close[-200:])
                        self.data['IHSG_MA50'] = ma50
                        self.data['IHSG_MA200'] = ma200
                        self.data['IHSG_Close'] = current_price

                        if current_price > ma50 > ma200:
                            trend = "BULLISH (di atas MA50 & MA200)"
                        elif current_price < ma50 < ma200:
                            trend = "BEARISH (di bawah MA50 & MA200)"
                        elif current_price > ma50:
                            trend = "NETRAL (di atas MA50, di bawah MA200)"
                        else:
                            trend = "NETRAL (di bawah MA50, di atas MA200)"
                    else:
                        trend = "TIDAK TERSEDIA"

                    self.detail[name] = {
                        'current_price': current_price,
                        'price_1w_ago': price_1w_ago,
                        'price_1m_ago': price_1m_ago,
                        'price_3m_ago': price_3m_ago,
                        'price_1y_ago': price_1y_ago,
                        'momentum_1w': round(momentum_1w, 2),
                        'momentum_1m': round(momentum_1m, 2),
                        'momentum_3m': round(momentum_3m, 2),
                        'momentum_1y': round(momentum_1y, 2),
                        'trend': trend,
                        'data_length': len(df)
                    }

                    self.data[name] = df
                    self.momentum[name] = round(momentum_1m, 2)
                    self.prices[name] = round(current_price, 2)
                    self.status[name] = "OK"

                    print(f"   ✅ Status: TERSEDIA")
                    print(f"   Harga Saat Ini: {self.get_price_str(name)}")
                    print(f"   Momentum 1w: {momentum_1w:+.2f}%")
                    print(f"   Momentum 1m: {momentum_1m:+.2f}%")
                    print(f"   Momentum 3m: {momentum_3m:+.2f}%")
                    print(f"   Momentum 1y: {momentum_1y:+.2f}%")
                    if name == "IHSG":
                        print(f"   MA50: {ma50:,.2f}")
                        print(f"   MA200: {ma200:,.2f}")
                        print(f"   Trend: {trend}")

            except Exception as e:
                self.status[name] = "ERROR"
                self.momentum[name] = 0.0
                self.prices[name] = 0.0
                print(f"   ❌ Status: ERROR - {str(e)[:50]}")
                time.sleep(0.5)

        print("\n" + "="*80)
        print("✅ SEMUA INDEKS GLOBAL SELESAI DI-FETCH")
        print("="*80)

    def print_detailed_report(self) -> None:
        if not self.detail:
            print("\n📊 Tidak ada data indeks untuk ditampilkan")
            return

        print("\n" + "="*100)
        print("📊 LAPORAN DETAIL INDEKS GLOBAL")
        print("="*100)

        data = []
        for name, detail in self.detail.items():
            data.append([
                name,
                self.get_price_str(name),
                f"{detail['momentum_1w']:+.2f}%",
                f"{detail['momentum_1m']:+.2f}%",
                f"{detail['momentum_3m']:+.2f}%",
                f"{detail['momentum_1y']:+.2f}%",
                detail.get('trend', 'N/A'),
                self.status[name]
            ])

        headers = ["Indeks", "Harga", "1W", "1M", "3M", "1Y", "Trend", "Status"]
        print(tabulate(data, headers=headers, tablefmt='grid'))

    def get_momentum(self, name: str) -> float:
        return self.momentum.get(name, 0.0)

    def get_price(self, name: str) -> float:
        return self.prices.get(name, 0.0)

    def get_price_str(self, name: str) -> str:
        price = self.get_price(name)
        if price == 0:
            return "N/A"

        if name in ["IHSG", "DOWJONES"]:
            return f"{price:,.2f}"
        elif name == "USDIDR":
            return f"Rp {price:,.0f}"
        elif name == "OIL":
            return f"US$ {price:.2f}"
        elif name == "GOLD":
            return f"US$ {price:.2f}"
        return f"{price:.2f}"

    def get_trend(self, name: str) -> str:
        mom = self.get_momentum(name)
        if mom > 0.5:
            return "🟢 BULLISH"
        elif mom < -0.5:
            return "🔴 BEARISH"
        else:
            return "🟡 NETRAL"

    def is_ihsg_bullish(self) -> bool:
        if 'IHSG_Close' in self.data and 'IHSG_MA200' in self.data:
            return self.data['IHSG_Close'] > self.data['IHSG_MA200']
        return True

# =============================================================================
# 7. DATA WAREHOUSE
# =============================================================================

class DataWarehouse:
    """Menyimpan data historis LENGKAP untuk semua saham"""

    def __init__(self, warehouse_dir: str = 'data_warehouse', min_days: int = 400):
        self.warehouse_dir = warehouse_dir
        self.min_days = min_days
        os.makedirs(warehouse_dir, exist_ok=True)

        self.stats = {
            'total_symbols': 0,
            'downloaded': 0,
            'failed': 0,
            'cached': 0,
            'filtered_min_days': 0
        }

    def download_complete_history(
        self,
        symbols: List[str],
        start_date: str = '2018-01-01',
        end_date: str = '2026-12-31',
        force_refresh: bool = False
    ) -> Dict[str, pd.DataFrame]:
        print("\n" + "="*80)
        print("🗄️  DATA WAREHOUSE - DOWNLOAD HISTORIS LENGKAP")
        print("="*80)
        print(f"Periode: {start_date} hingga {end_date}")
        print(f"Total saham: {len(symbols)}")
        print(f"Minimal hari: {self.min_days} (saham dengan data kurang akan difilter)")

        results = {}
        self.stats['total_symbols'] = len(symbols)

        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)

        for i, symbol in enumerate(symbols):
            cache_file = f"{self.warehouse_dir}/{symbol}_full.parquet"

            if os.path.exists(cache_file) and not force_refresh:
                try:
                    df = pd.read_parquet(cache_file)

                    if len(df) >= self.min_days and df.index[0] <= start_dt and df.index[-1] >= end_dt:
                        results[symbol] = df
                        self.stats['cached'] += 1
                    else:
                        self.stats['filtered_min_days'] += 1

                    if (i + 1) % 50 == 0:
                        print(f"   Progress: {i+1}/{len(symbols)} - {len(results)} dimuat (cache)")
                    continue
                except Exception:
                    pass

            try:
                ticker = f"{symbol}.JK"
                print(f"   Downloading {symbol} ({i+1}/{len(symbols)})...", end=" ")

                df = yf.download(
                    ticker,
                    start=start_date,
                    end=end_date,
                    interval="1d",
                    auto_adjust=True,
                    progress=False,
                    timeout=30
                )

                time.sleep(0.5)

                if df.empty:
                    print("❌ GAGAL (data kosong)")
                    self.stats['failed'] += 1
                    continue

                df = normalize_columns(df)

                if len(df) < self.min_days:
                    print(f"❌ GAGAL (hanya {len(df)} hari, minimal {self.min_days})")
                    self.stats['filtered_min_days'] += 1
                    continue

                df.to_parquet(cache_file)
                results[symbol] = df
                self.stats['downloaded'] += 1
                print(f"✅ {len(df)} hari")

            except Exception as e:
                print(f"❌ ERROR: {str(e)[:50]}")
                self.stats['failed'] += 1
                time.sleep(1)

        print("\n" + "="*80)
        print("📊 RINGKASAN DATA WAREHOUSE")
        print("="*80)
        print(f"Total saham: {self.stats['total_symbols']}")
        print(f"Berhasil dimuat: {len(results)}")
        print(f"  - Dari cache: {self.stats['cached']}")
        print(f"  - Download baru: {self.stats['downloaded']}")
        print(f"  - Difilter (< {self.min_days} hari): {self.stats['filtered_min_days']}")
        print(f"Gagal: {self.stats['failed']}")
        print("="*80)

        return results

    def get_all_valid_symbols(self) -> List[str]:
        symbols = []
        for f in os.listdir(self.warehouse_dir):
            if f.endswith('_full.parquet'):
                symbol = f.replace('_full.parquet', '')
                try:
                    df = pd.read_parquet(os.path.join(self.warehouse_dir, f))
                    if len(df) >= self.min_days:
                        symbols.append(symbol)
                except Exception:
                    continue
        return symbols

    def get_data(self, symbol: str) -> Optional[pd.DataFrame]:
        cache_file = f"{self.warehouse_dir}/{symbol}_full.parquet"

        if not os.path.exists(cache_file):
            return None

        try:
            df = pd.read_parquet(cache_file)
            if len(df) >= self.min_days:
                return df
            return None
        except Exception:
            return None

    def get_all_data(self, max_symbols: int = None) -> Dict[str, pd.DataFrame]:
        results = {}
        symbols = self.get_all_valid_symbols()

        if max_symbols:
            symbols = symbols[:max_symbols]

        for symbol in symbols:
            df = self.get_data(symbol)
            if df is not None:
                results[symbol] = df

        return results

    def print_warehouse_stats(self) -> None:
        symbols = self.get_all_valid_symbols()

        print("\n" + "="*80)
        print("🗄️  DATA WAREHOUSE STATISTICS")
        print("="*80)
        print(f"Total saham valid (≥{self.min_days} hari): {len(symbols)}")

        if symbols:
            sample = symbols[0]
            df = pd.read_parquet(f"{self.warehouse_dir}/{sample}_full.parquet")
            print(f"Rentang tanggal: {df.index[0].date()} hingga {df.index[-1].date()}")
            print(f"Rata-rata hari per saham: {len(df)} hari")

        print("="*80)

# =============================================================================
# 8. UTILITY FUNCTIONS
# =============================================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

def calculate_spread_pct(df: pd.DataFrame) -> float:
    try:
        spread = ((df['High'] - df['Low']) / df['Close']).tail(10).mean() * 100
        return float(spread)
    except Exception:
        return 999.0

def calculate_return(series: pd.Series, period: int = 5) -> float:
    if len(series) < period + 1:
        return 0.0
    return (series.iloc[-1] / series.iloc[-period-1] - 1) * 100

# =============================================================================
# 9. RISK MANAGER
# =============================================================================

class RiskManager:
    def __init__(
        self,
        modal: float,
        risk_per_trade_pct: float = 0.5,
        max_risk_portfolio_pct: float = 3.0,
        max_lot_per_position: int = 5
    ):
        self.modal = modal
        self.risk_per_trade_pct = risk_per_trade_pct
        self.max_risk_portfolio_pct = max_risk_portfolio_pct
        self.max_lot_per_position = max_lot_per_position

        self.risk_per_trade_rp = modal * (risk_per_trade_pct / 100)
        self.max_risk_portfolio_rp = modal * (max_risk_portfolio_pct / 100)

        self.current_positions = []
        self.current_risk_rp = 0.0

    def calculate_lot(
        self,
        close: float,
        atr: float,
        symbol: str = None
    ):
        if atr <= 0 or close <= 0:
            return None, None, None

        risk_per_lot = atr * 100
        raw_lot = self.risk_per_trade_rp / risk_per_lot
        lot = int(raw_lot)

        lot = min(lot, self.max_lot_per_position)
        max_lot_by_modal = int(self.modal / (close * 100))
        lot = min(lot, max_lot_by_modal)

        if lot < 1:
            return None, None, None

        cost = lot * 100 * close
        risk_amount = lot * risk_per_lot

        return lot, cost, risk_amount

    def can_add_position(self, risk_amount: float):
        if self.current_risk_rp + risk_amount > self.max_risk_portfolio_rp:
            return False, f"Portfolio risk limit: Rp {self.max_risk_portfolio_rp:,.0f}"
        return True, "OK"

    def add_position(self, symbol: str, lot: int, entry_price: float, risk_amount: float) -> None:
        position = {
            'symbol': symbol,
            'lot': lot,
            'entry_price': entry_price,
            'risk_amount': risk_amount,
            'entry_date': datetime.now()
        }
        self.current_positions.append(position)
        self.current_risk_rp += risk_amount

    def remove_position(self, symbol: str) -> None:
        for i, pos in enumerate(self.current_positions):
            if pos['symbol'] == symbol:
                self.current_risk_rp -= pos['risk_amount']
                self.current_positions.pop(i)
                break

    def get_portfolio_risk_pct(self) -> float:
        return (self.current_risk_rp / self.modal) * 100 if self.modal > 0 else 0

    def reset(self) -> None:
        self.current_positions = []
        self.current_risk_rp = 0.0

# =============================================================================
# 10. PORTFOLIO RISK CALCULATOR
# =============================================================================

class PortfolioRiskCalculator:
    def __init__(self, lookback_days: int = 60):
        self.lookback_days = lookback_days

    def calculate_correlation_matrix(
        self,
        symbols: List[str],
        price_data: Dict[str, pd.DataFrame]
    ) -> pd.DataFrame:
        returns_dict = {}

        for symbol in symbols:
            if symbol in price_data:
                df = price_data[symbol]
                if len(df) >= self.lookback_days:
                    returns = df['Close'].pct_change().tail(self.lookback_days)
                    returns_dict[symbol] = returns

        if not returns_dict:
            return pd.DataFrame()

        returns_df = pd.DataFrame(returns_dict)
        corr_matrix = returns_df.corr()
        return corr_matrix.fillna(0)

    def calculate_portfolio_var(
        self,
        positions: Dict[str, Dict],
        corr_matrix: pd.DataFrame,
        confidence: float = 0.95
    ):
        if not positions or corr_matrix.empty:
            return 0.0, 0.0, 0.0

        symbols = list(positions.keys())
        weights = []
        total_risk = sum([p['risk_amount'] for p in positions.values()])

        for symbol in symbols:
            if symbol in positions and symbol in corr_matrix.index:
                weight = positions[symbol]['risk_amount'] / total_risk if total_risk > 0 else 0
                weights.append(weight)

        if len(weights) < 2:
            z_score = {0.95: 1.645, 0.99: 2.326}[confidence]
            var_amount = total_risk * z_score
            return var_amount, total_risk, 1.0

        try:
            sub_corr = corr_matrix.loc[symbols, symbols].values
        except Exception:
            return total_risk * 1.645, total_risk, 1.0

        weights_array = np.array(weights)
        portfolio_variance = np.dot(np.dot(weights_array, sub_corr), weights_array)
        portfolio_std = np.sqrt(max(portfolio_variance, 0.001))

        z_score = {0.95: 1.645, 0.99: 2.326}[confidence]
        var_amount = total_risk * portfolio_std * z_score

        return var_amount, total_risk, portfolio_std

# =============================================================================
# 11. REALISTIC FEE CONFIG
# =============================================================================

class RealisticFeeConfig:
    BROKER_FEE_BUY = 0.0015
    BROKER_FEE_SELL = 0.0025
    EXCHANGE_FEE = 0.0001

    SLIPPAGE_MODEL = {
        'high': {'buy': 0.0005, 'sell': 0.0010},
        'medium': {'buy': 0.0010, 'sell': 0.0020},
        'low': {'buy': 0.0020, 'sell': 0.0030}
    }

    MIN_FEE_PER_TRANSACTION = 10000

    def __init__(self, liquidity: str = 'medium'):
        self.liquidity = liquidity
        self.slippage = self.SLIPPAGE_MODEL.get(liquidity, self.SLIPPAGE_MODEL['medium'])

    def calculate_buy_cost(self, price: float, lot: int) -> float:
        amount = price * 100 * lot
        broker_fee = amount * self.BROKER_FEE_BUY
        exchange_fee = amount * self.EXCHANGE_FEE
        slippage = amount * self.slippage['buy']
        total_fee = broker_fee + exchange_fee + slippage
        return max(total_fee, self.MIN_FEE_PER_TRANSACTION)

    def calculate_sell_cost(self, price: float, lot: int) -> float:
        amount = price * 100 * lot
        broker_fee = amount * self.BROKER_FEE_SELL
        exchange_fee = amount * self.EXCHANGE_FEE
        slippage = amount * self.slippage['sell']
        total_fee = broker_fee + exchange_fee + slippage
        return max(total_fee, self.MIN_FEE_PER_TRANSACTION)

    def calculate_round_trip(
        self,
        entry_price: float,
        exit_price: float,
        lot: int
    ):
        buy_cost = self.calculate_buy_cost(entry_price, lot)
        sell_cost = self.calculate_sell_cost(exit_price, lot)
        total_fee = buy_cost + sell_cost

        buy_amount = entry_price * 100 * lot
        sell_amount = exit_price * 100 * lot

        gross_profit = sell_amount - buy_amount
        net_profit = gross_profit - total_fee
        net_return_pct = (net_profit / buy_amount) * 100 if buy_amount > 0 else 0

        return total_fee, net_profit, net_return_pct

# =============================================================================
# 12. ENTRY DELAY SIMULATOR
# =============================================================================

class EntryDelaySimulator:
    def __init__(self, max_delay: int = 2):
        self.max_delay = max_delay

    def simulate_entry(
        self,
        df: pd.DataFrame,
        signal_idx: int,
        signal_price: float
    ):
        start_idx = signal_idx + 1
        end_idx = min(start_idx + self.max_delay, len(df) - 1)

        entry_candidates = []

        for idx in range(start_idx, end_idx + 1):
            row = df.iloc[idx]
            open_price = float(row['Open'])
            low_price = float(row['Low'])

            if open_price <= signal_price * 1.01:
                entry_candidates.append({
                    'idx': idx,
                    'price': open_price,
                    'delay': idx - signal_idx,
                    'type': 'open',
                    'reason': f'Open at {open_price:.0f}'
                })

            if low_price <= signal_price:
                entry_price = min(signal_price, open_price)
                entry_candidates.append({
                    'idx': idx,
                    'price': entry_price,
                    'delay': idx - signal_idx,
                    'type': 'limit',
                    'reason': f'Limit fill at {entry_price:.0f}'
                })

        if entry_candidates:
            return min(entry_candidates, key=lambda x: x['price'])

        return None

# =============================================================================
# 13. ACCURATE BACKTESTER
# =============================================================================

class AccurateBacktester:
    def __init__(self, initial_capital: float):
        self.initial_capital = initial_capital
        self.equity_curve = [{
            'date': None,
            'equity': initial_capital,
            'cash': initial_capital,
            'positions': {}
        }]
        self.trades = []
        self.fee_config = None

    def set_fee_config(self, fee_config: RealisticFeeConfig) -> None:
        self.fee_config = fee_config

    def calculate_trade_return(
        self,
        entry_price: float,
        exit_price: float,
        lot: int,
        include_fee: bool = True
    ):
        entry_value = entry_price * 100 * lot
        exit_value = exit_price * 100 * lot

        if include_fee and self.fee_config:
            fee, net_profit, net_return_pct = self.fee_config.calculate_round_trip(
                entry_price, exit_price, lot
            )
            return net_profit, net_return_pct, fee
        else:
            profit = exit_value - entry_value
            return_pct = (exit_value / entry_value - 1) * 100 if entry_value > 0 else 0
            return profit, return_pct, 0.0

    def add_trade(self, trade: Dict) -> float:
        self.trades.append(trade)

        last_equity = self.equity_curve[-1]['equity']
        new_equity = last_equity + trade['profit_rp']

        positions = self.equity_curve[-1]['positions'].copy()
        if trade['symbol'] in positions:
            del positions[trade['symbol']]

        self.equity_curve.append({
            'date': trade['exit_date'],
            'equity': new_equity,
            'cash': new_equity,
            'positions': positions,
            'trade': trade
        })

        return new_equity

    def calculate_metrics(self) -> Dict:
        if len(self.equity_curve) < 2:
            return {}

        equities = [e['equity'] for e in self.equity_curve]

        final_equity = equities[-1]
        total_return_pct = (final_equity / self.initial_capital - 1) * 100
        total_profit_rp = final_equity - self.initial_capital

        peak = equities[0]
        max_dd_rp = 0
        max_dd_pct = 0

        for eq in equities:
            if eq > peak:
                peak = eq
            else:
                dd_rp = peak - eq
                dd_pct = (dd_rp / peak) * 100 if peak > 0 else 0
                if dd_pct > max_dd_pct:
                    max_dd_pct = dd_pct
                    max_dd_rp = dd_rp

        if len(equities) > 1:
            returns_series = pd.Series(equities).pct_change().dropna()
            if len(returns_series) > 0 and returns_series.std() > 0:
                sharpe = (returns_series.mean() / returns_series.std()) * np.sqrt(252)
            else:
                sharpe = 0.0
        else:
            sharpe = 0.0

        if self.trades:
            profits = [t['profit_rp'] for t in self.trades]

            winning_trades = [p for p in profits if p > 0]
            losing_trades = [p for p in profits if p <= 0]

            win_rate = len(winning_trades) / len(self.trades) * 100 if self.trades else 0

            avg_win = np.mean(winning_trades) if winning_trades else 0
            avg_loss = abs(np.mean(losing_trades)) if losing_trades else 0

            total_win = sum(winning_trades) if winning_trades else 0
            total_loss = abs(sum(losing_trades)) if losing_trades else 0
            profit_factor = total_win / total_loss if total_loss > 0 else float('inf')
        else:
            win_rate = 0
            avg_win = 0
            avg_loss = 0
            profit_factor = 0

        return {
            'total_return_pct': round(total_return_pct, 2),
            'total_profit_rp': round(total_profit_rp),
            'max_drawdown_pct': round(max_dd_pct, 2),
            'max_drawdown_rp': round(max_dd_rp),
            'sharpe_ratio': round(sharpe, 2),
            'win_rate': round(win_rate, 1),
            'profit_factor': round(profit_factor, 2),
            'avg_win_rp': round(avg_win),
            'avg_loss_rp': round(avg_loss),
            'total_trades': len(self.trades),
            'final_equity': round(final_equity)
        }

# =============================================================================
# 14. BASE STRATEGY ENGINE
# =============================================================================

class BaseStrategyEngine:
    def __init__(self, config: Any, global_fetcher: GlobalIndicesFetcher):
        self.config = config
        self.global_fetcher = global_fetcher
        self.risk_manager = None

    def set_risk_manager(self, risk_manager: RiskManager) -> None:
        self.risk_manager = risk_manager

    def get_signal(self, symbol: str, df: pd.DataFrame):
        raise NotImplementedError

# =============================================================================
# 15. MODAL ADAPTER
# =============================================================================

class ModalAdapter:
    def __init__(self, modal: float):
        self.modal = modal

        if modal < 50000:
            self.kategori = "ULTRA_MIKRO"
        elif modal < 100000:
            self.kategori = "MIKRO"
        elif modal < 500000:
            self.kategori = "RETAIL"
        elif modal < 2000000:
            self.kategori = "MENENGAH"
        else:
            self.kategori = "INSTITUSI"

        self.max_harga_beli = modal / 100

    def get_filter_harga(self):
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 50, min(500, self.max_harga_beli)
        elif self.kategori == "RETAIL":
            return 100, min(2000, self.max_harga_beli)
        elif self.kategori == "MENENGAH":
            return 200, min(5000, self.max_harga_beli)
        else:
            return 500, 50000

    def get_filter_volume(self):
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 10000, 50000
        elif self.kategori == "RETAIL":
            return 25000, 100000
        elif self.kategori == "MENENGAH":
            return 50000, 250000
        else:
            return 100000, 500000

    def get_filter_spread(self) -> float:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 5.0
        elif self.kategori == "RETAIL":
            return 4.0
        elif self.kategori == "MENENGAH":
            return 3.0
        else:
            return 2.0

    def get_filter_min_history(self) -> int:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 50
        elif self.kategori == "RETAIL":
            return 100
        elif self.kategori == "MENENGAH":
            return 150
        else:
            return 200

    def get_quality_filter(self) -> str:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO", "RETAIL"]:
            return "ALL"
        elif self.kategori == "MENENGAH":
            return "LQ45"
        else:
            return "QUALITY"

    def get_entry_tolerance(self) -> float:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 10.0
        elif self.kategori == "RETAIL":
            return 7.0
        elif self.kategori == "MENENGAH":
            return 5.0
        else:
            return 2.0

    def get_trend_tolerance(self) -> float:
        if self.kategori in ["ULTRA_MIKRO", "MIKRO"]:
            return 0.90
        elif self.kategori == "RETAIL":
            return 0.95
        elif self.kategori == "MENENGAH":
            return 0.98
        else:
            return 1.00

    def print_info(self) -> None:
        print(f"\n📊 ADAPTASI FILTER UNTUK MODAL: {self.kategori}")
        print(f"   Max harga per saham: Rp {self.max_harga_beli:,.0f}")
        print(f"   Filter harga: Rp {self.get_filter_harga()[0]} - {self.get_filter_harga()[1]}")
        print(f"   Filter volume: {self.get_filter_volume()[0]} - {self.get_filter_volume()[1]} lembar")
        print(f"   Max spread: {self.get_filter_spread()}%")
        print(f"   Minimal history: {self.get_filter_min_history()} hari")
        print(f"   Quality filter: {self.get_quality_filter()}")
        print(f"   Entry tolerance: {self.get_entry_tolerance()}% dari MA50")
        print(f"   Trend tolerance: {self.get_trend_tolerance()*100:.0f}% dari MA200")
        print(f"\n⚠️  RISK MANAGEMENT TETAP STABIL: 0.5% per trade, 3% portfolio")

# =============================================================================
# 16. INVESTASI ENGINE
# =============================================================================

class InvestasiEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)
        self.quality_stocks = QUALITY_STOCKS
        self.modal_adapter = ModalAdapter(config.MODAL)

    def calculate_atr(self, df: pd.DataFrame, period: int = 14):
        try:
            high = df['High']
            low = df['Low']
            close = df['Close']

            tr1 = high - low
            tr2 = (high - close.shift()).abs()
            tr3 = (low - close.shift()).abs()

            tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
            atr = tr.rolling(window=period).mean().iloc[-1]

            return float(atr)
        except Exception:
            return None

    def get_signal(self, symbol: str, df: pd.DataFrame):
        quality_filter = self.modal_adapter.get_quality_filter()
        if quality_filter == "QUALITY" and symbol not in self.quality_stocks:
            return None
        elif quality_filter == "LQ45" and symbol not in QUALITY_STOCKS[:45]:
            return None

        min_history = self.modal_adapter.get_filter_min_history()
        if len(df) < min_history:
            return None

        close = df['Close']
        ma200 = close.rolling(window=200).mean()
        ma50 = close.rolling(window=50).mean()

        if len(ma200) < 200 or len(ma50) < 50:
            return None

        current_price = float(close.iloc[-1])
        current_ma200 = float(ma200.iloc[-1])
        current_ma50 = float(ma50.iloc[-1])

        trend_tolerance = self.modal_adapter.get_trend_tolerance()
        if current_price < current_ma200 * trend_tolerance:
            return None

        price_to_ma50 = (current_price / current_ma50 - 1) * 100
        entry_tolerance = self.modal_adapter.get_entry_tolerance()

        if price_to_ma50 > entry_tolerance:
            return None

        if price_to_ma50 < -entry_tolerance * 1.5:
            return None

        min_price, max_price = self.modal_adapter.get_filter_harga()
        if current_price < min_price or current_price > max_price:
            return None

        min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
        last_volume = int(df['Volume'].iloc[-1])
        avg_volume_20 = df['Volume'].tail(20).mean()

        if last_volume < min_vol or avg_volume_20 < min_avg_vol:
            return None

        spread = calculate_spread_pct(df)
        max_spread = self.modal_adapter.get_filter_spread()
        if spread > max_spread:
            return None

        atr = self.calculate_atr(df)
        if atr is None or atr <= 0:
            atr = current_price * 0.02

        lot, cost, risk_amount = self.risk_manager.calculate_lot(current_price, atr, symbol)

        if lot is None:
            return None

        can_add, reason = self.risk_manager.can_add_position(risk_amount)
        if not can_add:
            return None

        stop_loss = current_ma200 * trend_tolerance * 0.95

        sector = get_sector(symbol)
        risk_pct = (risk_amount / self.risk_manager.modal) * 100

        return {
            'Symbol': symbol,
            'Sector': sector,
            'Price': int(current_price),
            'MA50': int(current_ma50),
            'MA200': int(current_ma200),
            'To_MA50': f"{price_to_ma50:.1f}%",
            'Stop_Loss': int(stop_loss),
            'Take_Profit': 'HOLD',
            'Lot': lot,
            'Cost': int(cost),
            'Risk_Amount': int(risk_amount),
            'Risk_Pct': round(risk_pct, 1),
            'ATR': int(atr),
            'Reasons': f'Pullback ke MA50 ({price_to_ma50:.1f}%), di atas MA200',
            'Chart': 'BULLISH' if current_price > current_ma200 else 'NEUTRAL'
        }

# =============================================================================
# 17. SWING ENGINE
# =============================================================================

class SwingEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)
        self.rsi_period = getattr(config, 'RSI_PERIOD', 14)
        self.ma_short = getattr(config, 'MA_SHORT', 20)
        self.ma_long = getattr(config, 'MA_LONG', 50)
        self.ma200_period = getattr(config, 'MA200_PERIOD', 200)
        self.atr_period = getattr(config, 'ATR_PERIOD', 14)
        self.sl_multiplier = getattr(config, 'SL_MULTIPLIER', 1.5)
        self.tp_multiplier = getattr(config, 'TP_MULTIPLIER', 2.5)
        self.volume_boost = getattr(config, 'VOLUME_BOOST', 1.5)
        self.min_rr = getattr(config, 'MIN_RR', 1.0)
        self.min_ev_pct = getattr(config, 'MIN_EV_PCT', 2.0)

        self.modal_adapter = ModalAdapter(config.MODAL)

    def calculate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']

            delta = close.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.rolling(window=self.rsi_period, min_periods=self.rsi_period).mean()
            avg_loss = loss.rolling(window=self.rsi_period, min_periods=self.rsi_period).mean()
            rs = avg_gain / (avg_loss + 1e-6)
            out['RSI'] = 100 - (100 / (1 + rs))

            out['MA20'] = close.rolling(window=self.ma_short, min_periods=self.ma_short).mean()
            out['MA50'] = close.rolling(window=self.ma_long, min_periods=self.ma_long).mean()
            out['MA200'] = close.rolling(window=self.ma200_period, min_periods=self.ma200_period).mean()
            out['MA_Trend'] = (out['MA20'] > out['MA50']).astype(float)

            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()).abs(), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.atr_period, min_periods=self.atr_period).mean()

            out['Volume_MA'] = volume.rolling(window=20, min_periods=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)

            out = out.replace([np.inf, -np.inf], np.nan)
            out = out.dropna()
            return out

        except Exception:
            return pd.DataFrame()

    def get_signal(self, symbol: str, df: pd.DataFrame):
        min_history = self.modal_adapter.get_filter_min_history()
        if len(df) < min_history:
            return None

        df_feat = self.calculate_features(df)
        if len(df_feat) < self.ma200_period:
            return None

        latest = df_feat.iloc[-1]
        close = float(latest['Close'])

        min_price, max_price = self.modal_adapter.get_filter_harga()
        if close < min_price or close > max_price:
            return None

        min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
        last_volume = int(df['Volume'].iloc[-1])
        avg_volume_20 = df['Volume'].tail(20).mean()

        if last_volume < min_vol or avg_volume_20 < min_avg_vol:
            return None

        spread = calculate_spread_pct(df)
        max_spread = self.modal_adapter.get_filter_spread()
        if spread > max_spread:
            return None

        ma200 = float(latest['MA200']) if not pd.isna(latest['MA200']) else close
        trend_tolerance = self.modal_adapter.get_trend_tolerance()
        if close < ma200 * trend_tolerance:
            return None

        rsi = float(latest['RSI']) if not pd.isna(latest['RSI']) else 50
        volume_ratio = float(latest['Volume_Ratio']) if not pd.isna(latest['Volume_Ratio']) else 1
        atr = float(latest['ATR']) if not pd.isna(latest['ATR']) else close * 0.02
        ma20 = float(latest['MA20']) if not pd.isna(latest['MA20']) else close
        ma50 = float(latest['MA50']) if not pd.isna(latest['MA50']) else close

        score = 0
        if rsi < 30:
            score += 3
        elif rsi < 40:
            score += 1

        if volume_ratio > self.volume_boost:
            score += 2
        elif volume_ratio > 1.2:
            score += 1

        if ma20 > ma50:
            score += 1

        sl = close - (atr * self.sl_multiplier)
        tp = close + (atr * self.tp_multiplier)

        fraction = 5 if close < 100 else 10 if close < 500 else 25
        sl = round(sl / fraction) * fraction
        tp = round(tp / fraction) * fraction

        if sl >= close or tp <= close:
            return None

        risk = close - sl
        reward = tp - close
        rr = reward / risk

        if rr < self.min_rr:
            return None

        prob_up = 0.5 + (score * 0.02)
        prob_up = min(prob_up, 0.7)
        expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
        ev_pct = (expected_value / close) * 100

        if ev_pct < self.min_ev_pct:
            return None

        lot, cost, risk_amount = self.risk_manager.calculate_lot(close, atr, symbol)

        if lot is None:
            return None

        can_add, reason = self.risk_manager.can_add_position(risk_amount)
        if not can_add:
            return None

        sector = get_sector(symbol)
        risk_pct = (risk_amount / self.risk_manager.modal) * 100

        return {
            'Symbol': symbol,
            'Sector': sector,
            'Price': int(close),
            'RSI': round(rsi, 1),
            'Stop_Loss': int(sl),
            'Take_Profit': int(tp),
            'R/R': round(rr, 2),
            'Prob_Up': round(prob_up, 3),
            'EV_Pct': round(ev_pct, 2),
            'Score': score,
            'ATR': int(atr),
            'Lot': lot,
            'Cost': int(cost),
            'Risk_Amount': int(risk_amount),
            'Risk_Pct': round(risk_pct, 1),
            'Volume': f"{volume_ratio:.1f}x",
            'Reasons': f"RSI {rsi:.0f}, Vol {volume_ratio:.1f}x",
            'Chart': "BULLISH" if ma20 > ma50 else "NETRAL"
        }

# =============================================================================
# 18. INTRADAY LIQUID ENGINE
# =============================================================================

class IntradayLiquidEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)
        self.breakout_period = getattr(config, 'BREAKOUT_PERIOD', 10)
        self.ma_short = getattr(config, 'MA_SHORT', 20)
        self.ma_long = getattr(config, 'MA_LONG', 50)
        self.atr_period = getattr(config, 'ATR_PERIOD', 14)
        self.volume_breakout = getattr(config, 'VOLUME_BREAKOUT', 1.2)
        self.sl_multiplier = getattr(config, 'SL_MULTIPLIER', 0.8)
        self.tp_multiplier = getattr(config, 'TP_MULTIPLIER', 1.2)
        self.min_rr = getattr(config, 'MIN_RR', 1.0)
        self.min_ev_pct = getattr(config, 'MIN_EV_PCT', 2.0)

        self.modal_adapter = ModalAdapter(config.MODAL)

    def calculate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']

            out['Highest_High'] = high.shift(1).rolling(window=self.breakout_period).max()
            out['MA_Short'] = close.rolling(window=self.ma_short).mean()
            out['MA_Long'] = close.rolling(window=self.ma_long).mean()
            out['MA_Alignment'] = (out['MA_Short'] > out['MA_Long']).astype(int)
            out['Volume_MA'] = volume.rolling(window=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)
            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()).abs(), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.atr_period).mean()
            out['Body'] = abs(close - out['Open'])
            out['Range'] = high - low
            out['Body_Ratio'] = out['Body'] / (out['Range'] + 1e-6)

            out = out.replace([np.inf, -np.inf], np.nan)
            out = out.dropna()
            return out

        except Exception:
            return pd.DataFrame()

    def check_breakout(self, row: pd.Series) -> bool:
        if pd.isna(row['Highest_High']) or pd.isna(row['Close']):
            return False
        if row['Close'] <= row['Highest_High']:
            return False
        if row['Volume_Ratio'] < self.volume_breakout:
            return False
        if row['MA_Alignment'] != 1:
            return False
        return True

    def get_signal(self, symbol: str, df: pd.DataFrame):
        min_history = self.modal_adapter.get_filter_min_history()
        if len(df) < min_history:
            return None

        df_feat = self.calculate_features(df)
        if len(df_feat) < 30:
            return None

        latest = df_feat.iloc[-1]
        close = float(latest['Close'])

        min_price, max_price = self.modal_adapter.get_filter_harga()
        if close < min_price or close > max_price:
            return None

        min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
        last_volume = int(df['Volume'].iloc[-1])
        avg_volume_20 = df['Volume'].tail(20).mean()

        if last_volume < min_vol or avg_volume_20 < min_avg_vol:
            return None

        spread = calculate_spread_pct(df)
        max_spread = self.modal_adapter.get_filter_spread()
        if spread > max_spread:
            return None

        if not self.check_breakout(latest):
            return None

        atr = float(latest['ATR']) if not pd.isna(latest['ATR']) else close * 0.02
        atr = max(atr, close * 0.005)

        sl = close - (atr * self.sl_multiplier)
        tp = close + (atr * self.tp_multiplier)

        fraction = 5 if close < 100 else 10 if close < 500 else 25
        sl = round(sl / fraction) * fraction
        tp = round(tp / fraction) * fraction

        if sl >= close or tp <= close:
            return None

        risk = close - sl
        reward = tp - close
        rr = reward / risk

        if rr < self.min_rr:
            return None

        score = 5
        if latest['Volume_Ratio'] > 1.5:
            score += 1
        if latest['Body_Ratio'] > 0.6:
            score += 1

        prob_up = 0.5 + (score * 0.02)
        prob_up = min(prob_up, 0.7)
        expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
        ev_pct = (expected_value / close) * 100

        if ev_pct < self.min_ev_pct:
            return None

        lot, cost, risk_amount = self.risk_manager.calculate_lot(close, atr, symbol)

        if lot is None:
            return None

        can_add, reason = self.risk_manager.can_add_position(risk_amount)
        if not can_add:
            return None

        sector = get_sector(symbol)
        risk_pct = (risk_amount / self.risk_manager.modal) * 100

        return {
            'Symbol': symbol,
            'Sector': sector,
            'Price': int(close),
            'Stop_Loss': int(sl),
            'Take_Profit': int(tp),
            'R/R': round(rr, 2),
            'Prob_Up': round(prob_up, 3),
            'EV_Pct': round(ev_pct, 2),
            'Score': score,
            'ATR': int(atr),
            'Lot': lot,
            'Cost': int(cost),
            'Risk_Amount': int(risk_amount),
            'Risk_Pct': round(risk_pct, 1),
            'Volume': f"{latest['Volume_Ratio']:.1f}x",
            'Reasons': f"Breakout {self.breakout_period}, Vol {latest['Volume_Ratio']:.1f}x",
            'Chart': "BULLISH"
        }

# =============================================================================
# 19. INTRADAY GORENGAN ENGINE
# =============================================================================

class IntradayGorenganEngine(BaseStrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)
        self.atr_period = getattr(config, 'ATR_PERIOD', 14)
        self.min_volume_spike = getattr(config, 'MIN_VOLUME_SPIKE', 2.0)
        self.min_rr = getattr(config, 'MIN_RR', 1.0)
        self.min_ev_pct = getattr(config, 'MIN_EV_PCT', 1.5)

        self.modal_adapter = ModalAdapter(config.MODAL)

    def calculate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']
            open_price = out['Open']

            out['Highest_High_5'] = high.shift(1).rolling(window=5).max()
            out['Volume_MA'] = volume.rolling(window=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)
            out['Turnover'] = close * volume
            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()).abs(), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.atr_period).mean()
            out['Body'] = abs(close - open_price)
            out['Range'] = high - low
            out['Body_Ratio'] = out['Body'] / (out['Range'] + 1e-6)
            out['Day_Change'] = (close / open_price - 1) * 100

            out = out.replace([np.inf, -np.inf], np.nan)
            out = out.dropna()
            return out

        except Exception:
            return pd.DataFrame()

    def check_breakout(self, row: pd.Series, modal: float) -> bool:
        if pd.isna(row['Highest_High_5']) or pd.isna(row['Close']):
            return False
        if row['Close'] <= row['Highest_High_5']:
            return False
        if row['Volume_Ratio'] < self.min_volume_spike:
            return False
        min_turnover = modal * 5
        if row['Turnover'] < min_turnover:
            return False
        if row['Day_Change'] > 20:
            return False
        return True

    def get_signal(self, symbol: str, df: pd.DataFrame):
        min_history = self.modal_adapter.get_filter_min_history()
        if len(df) < min_history:
            return None

        df_feat = self.calculate_features(df)
        if len(df_feat) < 30:
            return None

        latest = df_feat.iloc[-1]
        close = float(latest['Close'])

        min_price, max_price = self.modal_adapter.get_filter_harga()
        if close < min_price or close > max_price:
            return None

        min_vol, min_avg_vol = self.modal_adapter.get_filter_volume()
        last_volume = int(df['Volume'].iloc[-1])
        avg_volume_20 = df['Volume'].tail(20).mean()

        if last_volume < min_vol or avg_volume_20 < min_avg_vol:
            return None

        spread = calculate_spread_pct(df)
        max_spread = self.modal_adapter.get_filter_spread()
        if spread > max_spread:
            return None

        if not self.check_breakout(latest, self.risk_manager.modal):
            return None

        atr = float(latest['ATR']) if not pd.isna(latest['ATR']) else close * 0.03
        atr = max(atr, close * 0.01)

        sl = close - (atr * 1.0)
        tp = close + (atr * 1.5)

        fraction = 5 if close < 100 else 10 if close < 500 else 25
        sl = round(sl / fraction) * fraction
        tp = round(tp / fraction) * fraction

        if sl >= close or tp <= close:
            return None

        risk = close - sl
        reward = tp - close
        rr = reward / risk

        if rr < self.min_rr:
            return None

        score = 5
        if latest['Volume_Ratio'] > 3:
            score += 2
        elif latest['Volume_Ratio'] > 2:
            score += 1
        if latest['Body_Ratio'] > 0.7:
            score += 1

        prob_up = 0.5 + (score * 0.02)
        prob_up = min(prob_up, 0.7)
        expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
        ev_pct = (expected_value / close) * 100

        if ev_pct < self.min_ev_pct:
            return None

        lot, cost, risk_amount = self.risk_manager.calculate_lot(close, atr, symbol)

        if lot is None:
            return None

        can_add, reason = self.risk_manager.can_add_position(risk_amount)
        if not can_add:
            return None

        sector = get_sector(symbol)
        risk_pct = (risk_amount / self.risk_manager.modal) * 100

        return {
            'Symbol': symbol,
            'Sector': sector,
            'Price': int(close),
            'Stop_Loss': int(sl),
            'Take_Profit': int(tp),
            'R/R': round(rr, 2),
            'Prob_Up': round(prob_up, 3),
            'EV_Pct': round(ev_pct, 2),
            'Score': score,
            'ATR': int(atr),
            'Volume_Spike': f"{latest['Volume_Ratio']:.1f}x",
            'Turnover': f"Rp {latest['Turnover']/1e6:.0f}Jt",
            'Lot': lot,
            'Cost': int(cost),
            'Risk_Amount': int(risk_amount),
            'Risk_Pct': round(risk_pct, 1),
            'Reasons': f"Breakout 5, Vol {latest['Volume_Ratio']:.1f}x",
            'Chart': "BULLISH"
        }

# =============================================================================
# 20. CONFIGURATION CLASSES
# =============================================================================

class SwingConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'swing'
        self.INTERVAL = "1d"
        self.PERIOD = "6mo"
        self.RSI_PERIOD = 14
        self.MA_SHORT = 20
        self.MA_LONG = 50
        self.MA200_PERIOD = 200
        self.ATR_PERIOD = 14
        self.SL_MULTIPLIER = 1.5
        self.TP_MULTIPLIER = 2.5
        self.VOLUME_BOOST = 1.5
        self.MIN_RR = 1.0
        self.MIN_EV_PCT = 2.0

class IntradayConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'intraday'
        self.INTERVAL = "1h"
        self.PERIOD = "1mo"
        self.BREAKOUT_PERIOD = 10
        self.VOLUME_BREAKOUT = 1.2
        self.MA_SHORT = 20
        self.MA_LONG = 50
        self.ATR_PERIOD = 14
        self.SL_MULTIPLIER = 0.8
        self.TP_MULTIPLIER = 1.2
        self.MIN_RR = 1.0
        self.MIN_EV_PCT = 2.0

class GorenganConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'gorengan'
        self.INTERVAL = "1h"
        self.PERIOD = "1mo"
        self.ATR_PERIOD = 14
        self.MIN_VOLUME_SPIKE = 2.0
        self.MIN_RR = 1.0
        self.MIN_EV_PCT = 1.5

class InvestasiConfig:
    def __init__(self, modal: float):
        self.MODAL = modal
        self.MODE = 'investasi'
        self.INTERVAL = "1d"
        self.PERIOD = "5y"

# =============================================================================
# 21. GOOGLE SHEETS EXPORTER
# =============================================================================

class GoogleSheetsExporter:
    """Export hasil scan ke Google Sheets - 1 sheet per engine"""

    def __init__(self):
        self.initialized = False
        self.spreadsheet = None
        self._url_printed = False

        # Template header per engine
        self.HEADERS = {
            'SWING ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'RSI',
                'Support', 'Resistance', 'Stop Loss', 'Take Profit',
                'R/R', 'Prob Up', 'EV %', 'Score', 'Volume', 'Inflow', 'Acc'
            ],
            'INTRADAY LIQUID ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'Volume Ratio',
                'Body Ratio', 'Upper Wick', 'Stop Loss', 'Take Profit',
                'R/R', 'Prob Up', 'EV %', 'Score', 'Inflow', 'Acc'
            ],
            'INTRADAY GORENGAN ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'Volume Spike',
                'Turnover', 'Day Change %', 'Stop Loss', 'Take Profit',
                'R/R', 'Prob Up', 'EV %', 'Score', 'Inflow', 'Acc'
            ],
            'INVESTASI ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'To MA50',
                'MA50', 'MA200', 'PER', 'PBV', 'Market Cap',
                'Stop Loss', 'Risk %', 'Lot', 'Cost', 'Quality', 'MA200+'
            ]
        }

    def _init_sheets(self):
        """Initialize Google Sheets API - TANPA FILE CREDENTIALS"""
        try:
            auth.authenticate_user()
            creds, _ = default()
            self.gc = gspread.authorize(creds)
            self.initialized = True
            print("✅ Berhasil konek ke Google Sheets")
            return True
        except Exception as e:
            print(f"❌ Gagal konek ke Google Sheets: {e}")
            return False

    def _get_or_create_spreadsheet(self, sheet_name):
        """Get atau create spreadsheet utama"""
        try:
            self.spreadsheet = self.gc.open(sheet_name)
            print(f"✅ Membuka spreadsheet: {sheet_name}")
        except:
            self.spreadsheet = self.gc.create(sheet_name)
            print(f"✅ Membuat spreadsheet baru: {sheet_name}")

            # Share dengan user (opsional)
            try:
                email = input("\n📧 Share spreadsheet dengan email (optional, tekan Enter untuk skip): ").strip()
                if email:
                    self.spreadsheet.share(email, perm_type='user', role='writer')
                    print(f"✅ Shared with {email}")
            except:
                pass

        return self.spreadsheet

    def _get_or_create_engine_sheet(self, engine_name):
        """Get atau create sheet untuk engine tertentu (1 sheet permanen)"""
        sheet_title = engine_name.replace(' ', '_').replace('ENGINE', '').strip()[:100]

        try:
            worksheet = self.spreadsheet.worksheet(sheet_title)
            print(f"✅ Membuka sheet: {sheet_title}")

            existing_headers = worksheet.row_values(1)
            if not existing_headers:
                headers = self.HEADERS.get(engine_name, ['Date', 'Timestamp', 'Symbol', 'Signal'])
                worksheet.append_row(headers)

        except:
            worksheet = self.spreadsheet.add_worksheet(
                title=sheet_title,
                rows=10000,
                cols=30
            )
            print(f"✅ Membuat sheet baru: {sheet_title}")

            headers = self.HEADERS.get(engine_name, ['Date', 'Timestamp', 'Symbol', 'Signal'])
            worksheet.append_row(headers)

        return worksheet

    def _format_signal_row(self, signal, engine_name):
        """Format signal berdasarkan engine"""
        today = datetime.now().strftime('%Y-%m-%d')
        timestamp = datetime.now().strftime('%H:%M:%S')

        if engine_name == 'SWING ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal.get('RSI', '-'),
                signal.get('Support', '-'),
                signal.get('Resistance', '-'),
                signal.get('Stop_Loss', '-'),
                signal.get('Take_Profit', '-'),
                signal.get('R/R', '-'),
                f"{signal.get('Prob_Up', 0):.0%}" if signal.get('Prob_Up') else '-',
                signal.get('EV_Pct', '-'),
                signal.get('Score', '-'),
                signal.get('Volume', '-'),
                signal.get('Inflow', '-'),
                signal.get('Acc', '-')
            ]

        elif engine_name == 'INTRADAY LIQUID ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal.get('Volume', '-'),
                signal.get('Body_Ratio', '-'),
                signal.get('Upper_Wick', '-'),
                signal.get('Stop_Loss', '-'),
                signal.get('Take_Profit', '-'),
                signal.get('R/R', '-'),
                f"{signal.get('Prob_Up', 0):.0%}" if signal.get('Prob_Up') else '-',
                signal.get('EV_Pct', '-'),
                signal.get('Score', '-'),
                signal.get('Inflow', '-'),
                signal.get('Acc', '-')
            ]

        elif engine_name == 'INTRADAY GORENGAN ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal.get('Volume_Spike', '-'),
                signal.get('Turnover', '-'),
                signal.get('Day_Change', '-'),
                signal.get('Stop_Loss', '-'),
                signal.get('Take_Profit', '-'),
                signal.get('R/R', '-'),
                f"{signal.get('Prob_Up', 0):.0%}" if signal.get('Prob_Up') else '-',
                signal.get('EV_Pct', '-'),
                signal.get('Score', '-'),
                signal.get('Inflow', '-'),
                signal.get('Acc', '-')
            ]

        elif engine_name == 'INVESTASI ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal['To_MA50'],
                signal.get('MA50', '-'),
                signal.get('MA200', '-'),
                signal.get('PER', 'N/A'),
                signal.get('PBV', 'N/A'),
                signal.get('MarketCap', 'N/A'),
                signal['Stop_Loss'],
                f"{signal.get('Risk%', 0)}%",
                signal['Lot'],
                signal['Cost'],
                signal.get('Quality', '-'),
                "⭐" if signal.get('MA200_Bonus', 0) else "-"
            ]

        else:
            return [today, timestamp, signal['Symbol'], str(signal)]

    def ensure_spreadsheet_exists(self):
        """Pastikan spreadsheet sudah ada dan tampilkan URL (meski 0 sinyal)"""
        if not self.initialized:
            if not self._init_sheets():
                return False

        if not self.spreadsheet:
            sheet_name = "IDX_Scanner_All_Engines"
            self._get_or_create_spreadsheet(sheet_name)

        if not self._url_printed:
            print(f"\n📊 Google Sheets URL: {self.spreadsheet.url}")
            self._url_printed = True

        return True

    def export_signals(self, signals, engine_name, modal):
        """Export sinyal ke Google Sheets"""
        if not signals:
            print(f"ℹ️ Tidak ada sinyal untuk {engine_name}")
            return False

        if not self.initialized:
            if not self._init_sheets():
                return False

        try:
            if not self.spreadsheet:
                sheet_name = "IDX_Scanner_All_Engines"
                self._get_or_create_spreadsheet(sheet_name)

            worksheet = self._get_or_create_engine_sheet(engine_name)

            for signal in signals:
                row = self._format_signal_row(signal, engine_name)
                worksheet.append_row(row)

            print(f"✅ Berhasil export {len(signals)} sinyal ke {engine_name} sheet")

            if not self._url_printed:
                print(f"\n📊 Google Sheets URL: {self.spreadsheet.url}")
                self._url_printed = True

            return True

        except Exception as e:
            print(f"❌ Gagal export ke Google Sheets: {e}")
            return False

# =============================================================================
# 22. PORTFOLIO GUIDE (FITUR BARU - 3 REKOMENDASI TERBAIK)
# =============================================================================

def print_portfolio_guide(signals, modal, risk_manager):
    """
    Menampilkan panduan portfolio dengan 3 rekomendasi terbaik
    """
    if not signals:
        return

    # Urutkan sinyal berdasarkan skor (jika ada) atau risk%
    if 'Score' in signals[0]:
        sorted_signals = sorted(signals, key=lambda x: x.get('Score', 0), reverse=True)
    elif 'Final_Score' in signals[0]:
        sorted_signals = sorted(signals, key=lambda x: x.get('Final_Score', 0), reverse=True)
    else:
        sorted_signals = sorted(signals, key=lambda x: x['Risk_Amount'] / x['Lot'] if x['Lot'] > 0 else 0, reverse=True)

    # Ambil 3 terbaik
    top_3 = sorted_signals[:3]

    print("\n" + "="*100)
    print("📊 PANDUAN PORTOFOLIO - 3 REKOMENDASI TERBAIK")
    print("="*100)
    print(f"Modal: Rp {modal:,}")
    print(f"Risk per trade: {risk_manager.risk_per_trade_pct}% (Rp {risk_manager.risk_per_trade_rp:,.0f})")
    print(f"Max portfolio risk: {risk_manager.max_risk_portfolio_pct}% (Rp {risk_manager.max_risk_portfolio_rp:,.0f})")
    print("-"*100)

    total_cost = 0
    total_risk = 0

    for i, signal in enumerate(top_3, 1):
        risk_amount = signal.get('Risk_Amount', 0)
        risk_pct = (risk_amount / modal) * 100 if modal > 0 else 0
        cost = signal.get('Cost', 0)
        sl = signal.get('Stop_Loss', 0)
        price = signal.get('Price', 0)

        # Hitung persentase modal untuk posisi ini
        modal_pct = (cost / modal) * 100 if modal > 0 else 0

        print(f"\n{i}. {signal['Symbol']} ({signal.get('Sector', 'OTHER')})")
        print(f"   Harga: Rp {price:,} | Lot: {signal.get('Lot', 1)} | Biaya: Rp {cost:,} ({modal_pct:.1f}% modal)")
        print(f"   Stop Loss: Rp {sl:,} | Risk: Rp {risk_amount:,} ({risk_pct:.1f}% modal)")
        print(f"   Target: {signal.get('Take_Profit', 'HOLD')} | R/R: {signal.get('R/R', '-')}")
        if 'To_MA50' in signal:
            print(f"   MA50: Rp {signal.get('MA50', '-')} | To MA50: {signal['To_MA50']}")
        if 'Quality' in signal:
            print(f"   Quality: {signal.get('Quality', '-')} (Score: {signal.get('Quality_Score', '-')})")
        print(f"   Alasan: {signal.get('Reasons', '-')}")

        total_cost += cost
        total_risk += risk_amount

    print("\n" + "-"*100)
    print(f"📈 TOTAL 3 POSISI:")
    print(f"   Total Biaya: Rp {total_cost:,} ({(total_cost/modal)*100:.1f}% modal)")
    print(f"   Total Risk: Rp {total_risk:,} ({(total_risk/modal)*100:.1f}% modal)")
    print(f"   Sisa Modal: Rp {modal - total_cost:,}")

    if total_risk > risk_manager.max_risk_portfolio_rp:
        print(f"\n⚠️  PERINGATAN: Total risk melebihi batas {risk_manager.max_risk_portfolio_pct}%!")
        print(f"   Kurangi lot atau pilih 1-2 posisi saja.")
    else:
        print(f"\n✅ Risk masih dalam batas aman (max {risk_manager.max_risk_portfolio_pct}%).")

    print("="*100)

# =============================================================================
# 23. PHASE 2 - BLOCK BOOTSTRAP MONTE CARLO (FIXED)
# =============================================================================

class BlockBootstrapMonteCarlo:
    def __init__(self, returns_series: pd.Series, block_size: int = 5):
        self.returns = returns_series.dropna()
        self.block_size = block_size
        self.blocks = self._create_blocks()

    def _create_blocks(self) -> List[np.ndarray]:
        blocks = []
        for i in range(0, len(self.returns) - self.block_size + 1, self.block_size):
            block = self.returns.iloc[i:i+self.block_size].values
            blocks.append(block)
        return blocks

    def run_simulation(
        self,
        initial_capital: float,
        n_simulations: int = 1000,
        n_trades: int = 100,
        risk_per_trade_pct: float = 0.5
    ) -> Dict:
        if not self.blocks:
            return {}

        results = []

        for sim in range(n_simulations):
            n_blocks_needed = (n_trades + self.block_size - 1) // self.block_size
            sampled_blocks = random.choices(self.blocks, k=n_blocks_needed)
            sampled_returns = np.concatenate(sampled_blocks)[:n_trades]

            equity = initial_capital
            peak = equity
            max_dd = 0
            loss_streak = 0
            trades_done = 0

            for r in sampled_returns:
                trade_risk = equity * (risk_per_trade_pct / 100)
                trade_return = r * (trade_risk / 100)
                equity += trade_return
                trades_done += 1

                if equity > peak:
                    peak = equity
                else:
                    dd = (peak - equity) / peak * 100
                    if dd > max_dd:
                        max_dd = dd

                if r < 0:
                    loss_streak += 1
                    if loss_streak >= 5:
                        break
                else:
                    loss_streak = 0

                if equity < initial_capital * 0.3:
                    break

            total_return_pct = (equity / initial_capital - 1) * 100
            results.append({
                'return': total_return_pct,
                'max_dd': max_dd,
                'trades': trades_done
            })

        returns = np.array([r['return'] for r in results])
        max_dds = np.array([r['max_dd'] for r in results])
        trades_counts = np.array([r['trades'] for r in results])

        summary = {
            'n_simulations': n_simulations,
            'n_trades_max': n_trades,
            'avg_trades_done': np.mean(trades_counts),
            'mean_return': np.mean(returns),
            'median_return': np.median(returns),
            'std_return': np.std(returns),
            'percentile_5': np.percentile(returns, 5),
            'percentile_25': np.percentile(returns, 25),
            'percentile_75': np.percentile(returns, 75),
            'percentile_95': np.percentile(returns, 95),
            'min_return': np.min(returns),
            'max_return': np.max(returns),
            'avg_max_dd': np.mean(max_dds),
            'max_dd_95': np.percentile(max_dds, 95),
            'probability_profit': np.mean(returns > 0) * 100,
            'probability_2x': np.mean(returns > 100) * 100,
            'probability_50pct_loss': np.mean(returns < -50) * 100,
            'expected_return_per_trade': np.mean(returns) / np.mean(trades_counts) if np.mean(trades_counts) > 0 else 0
        }

        self.results = results
        self.summary = summary
        self.returns_array = returns

        return summary

    def print_results(self) -> None:
        if not hasattr(self, 'summary'):
            print("\n📊 No simulation results")
            return

        print("\n" + "="*100)
        print("🎲 BLOCK BOOTSTRAP MONTE CARLO RESULTS (FIXED)")
        print("="*100)
        print(f"Number of simulations: {self.summary['n_simulations']:,}")
        print(f"Max trades per simulation: {self.summary['n_trades_max']}")
        print(f"Rata-rata trades terealisasi: {self.summary['avg_trades_done']:.1f}")
        print(f"Risk per trade: 0.5%")

        print("\n📈 RETURN DISTRIBUTION:")
        print(f"   Mean return: {self.summary['mean_return']:.2f}%")
        print(f"   Median return: {self.summary['median_return']:.2f}%")
        print(f"   Std deviation: {self.summary['std_return']:.2f}%")
        print(f"   Best case (95th percentile): {self.summary['percentile_95']:.2f}%")
        print(f"   Worst case (5th percentile): {self.summary['percentile_5']:.2f}%")

        print("\n📉 DRAWDOWN ANALYSIS:")
        print(f"   Rata-rata max drawdown: {self.summary['avg_max_dd']:.2f}%")
        print(f"   Max drawdown 95th percentile: {self.summary['max_dd_95']:.2f}%")

        print("\n🎯 PROBABILITIES:")
        print(f"   Probability of profit: {self.summary['probability_profit']:.1f}%")
        print(f"   Probability of 2x money: {self.summary['probability_2x']:.1f}%")
        print(f"   Probability of 50% loss: {self.summary['probability_50pct_loss']:.1f}%")

# =============================================================================
# 24. PHASE 2 - WALK-FORWARD VALIDATOR (SINGKAT)
# =============================================================================

class WalkForwardValidator:
    def __init__(self, data_dict: Dict[str, pd.DataFrame], start_date: str, end_date: str):
        self.data_dict = data_dict
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date)
        self.results = []

    def split_data_by_date(self, train_years: float = 1.0, test_months: float = 3.0) -> List[Dict]:
        windows = []
        current_start = self.start_date

        train_delta = timedelta(days=int(365 * train_years))
        test_delta = timedelta(days=int(30 * test_months))

        while current_start + train_delta + test_delta <= self.end_date:
            train_end = current_start + train_delta
            test_end = train_end + test_delta

            windows.append({
                'train_start': current_start,
                'train_end': train_end,
                'test_start': train_end,
                'test_end': test_end
            })
            current_start += test_delta

        return windows

    def filter_data_by_date(self, start: datetime, end: datetime) -> Dict[str, pd.DataFrame]:
        filtered = {}
        for symbol, df in self.data_dict.items():
            mask = (df.index >= start) & (df.index <= end)
            filtered_df = df.loc[mask].copy()
            if len(filtered_df) > 20:
                filtered[symbol] = filtered_df
        return filtered

    def run_validation(self, engine_class, base_config, param_grid, train_years=1.0, test_months=3.0):
        windows = self.split_data_by_date(train_years, test_months)
        print(f"\n📊 Walk-Forward Validation: {len(windows)} windows")

        for i, window in enumerate(windows):
            print(f"\n   Window {i+1}/{len(windows)}")
            train_data = self.filter_data_by_date(window['train_start'], window['train_end'])
            test_data = self.filter_data_by_date(window['test_start'], window['test_end'])

            if len(train_data) < 3 or len(test_data) < 3:
                print(f"   ⚠️  Insufficient data, skipping")
                continue

            self.results.append({
                'window': i+1,
                'train_start': window['train_start'],
                'train_end': window['train_end'],
                'test_start': window['test_start'],
                'test_end': window['test_end'],
                'train_sharpe': 0,
                'test_sharpe': 0,
                'train_trades': 0,
                'test_trades': 0
            })

        return self.results

# =============================================================================
# 25. PHASE 2 - VALIDATION SUITE
# =============================================================================

class ValidationSuite:
    def __init__(self, engine_class, base_config, data_dict: Dict[str, pd.DataFrame]):
        self.engine_class = engine_class
        self.base_config = base_config
        self.data_dict = data_dict

    def run_all(self, walk_forward_params=None, sensitivity_params=None, monte_carlo_params=None):
        results = {}

        if monte_carlo_params is None:
            monte_carlo_params = {
                'block_sizes': [3, 5],
                'n_simulations': 100,
                'n_trades': 100
            }

        print("\n" + "="*80)
        print("PHASE 2.3: BLOCK BOOTSTRAP MONTE CARLO (FIXED)")
        print("="*80)

        all_returns = []
        for symbol, df in self.data_dict.items():
            returns = df['Close'].pct_change().dropna() * 100
            all_returns.extend(returns.values[:1000])

        returns_series = pd.Series(all_returns)

        for block_size in monte_carlo_params['block_sizes']:
            print(f"\n📦 Block Size: {block_size} days")
            mc = BlockBootstrapMonteCarlo(returns_series, block_size=block_size)
            mc_results = mc.run_simulation(
                initial_capital=self.base_config.MODAL,
                n_simulations=monte_carlo_params['n_simulations'],
                n_trades=monte_carlo_params['n_trades'],
                risk_per_trade_pct=0.5
            )
            mc.print_results()
            mc.plot_distribution(save_path=f'monte_carlo_bs{block_size}_fixed.png')
            results[f'monte_carlo_bs{block_size}'] = mc_results

        return results

# =============================================================================
# 26. DATA WAREHOUSE INITIALIZATION
# =============================================================================

def initialize_data_warehouse():
    print("\n" + "="*80)
    print("🗄️  INISIALISASI DATA WAREHOUSE")
    print("="*80)
    print("Fungsi ini akan mendownload data historis LENGKAP untuk semua saham IDX")
    print("Periode: 2018-01-01 hingga 2026-12-31")
    print(f"Total saham: {len(STOCKBIT_UNIVERSE)}")
    print(f"Minimal hari: 400 (saham dengan data kurang akan difilter otomatis)")
    print("\n⚠️  PERINGATAN: Proses ini akan memakan waktu BEBERAPA JAM!")

    confirm = input("\nLanjutkan download? (y/n): ").strip().lower()
    if confirm != 'y':
        print("❌ Dibatalkan.")
        return None

    warehouse = DataWarehouse(warehouse_dir='data_warehouse', min_days=400)
    data = warehouse.download_complete_history(
        symbols=STOCKBIT_UNIVERSE,
        start_date='2018-01-01',
        end_date='2026-12-31'
    )

    warehouse.print_warehouse_stats()
    return data

# =============================================================================
# 27. PHASE 1 - MAIN PROGRAM (DENGAN PANDUAN PORTFOLIO)
# =============================================================================

def run_phase1(sheets_exporter):
    """Main program Phase 1 - Trading Scanner dengan Panduan Portfolio"""

    print("\n" + "="*60)
    print("🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE")
    print("   Modal-Adaptive Filters | Risk Stabil 0.5% per trade")
    print("   Export: Google Sheets | Panduan Portfolio 3 Terbaik")
    print("="*60)

    warehouse = DataWarehouse(warehouse_dir='data_warehouse', min_days=400)
    symbols = warehouse.get_all_valid_symbols()

    if not symbols:
        print("\n⚠️  Data warehouse kosong atau tidak ditemukan!")
        print("   Jalankan opsi 3 (Inisialisasi Data Warehouse) terlebih dahulu.")
        return

    print(f"\n📊 Data warehouse siap: {len(symbols)} saham dengan data >= 400 hari")

    print("\nPilih engine trading:")
    print("1. Swing Engine (Mingguan - Mean Reversion)")
    print("2. Intraday Liquid (1 jam - Momentum)")
    print("3. Intraday Gorengan (1 jam - Early Momentum)")
    print("4. Investasi Engine (Quality + Trend - Jangka Panjang)")

    while True:
        engine_choice = input("Pilihan (1/2/3/4): ").strip()
        if engine_choice in ['1', '2', '3', '4']:
            break
        print("❌ Pilih 1, 2, 3, atau 4")

    while True:
        try:
            modal_input = input("\nModal (Rp 10,000 - 5,000,000): ").strip()
            modal = int(modal_input.replace('.', '').replace(',', ''))
            if 10000 <= modal <= 5000000:
                break
            print("❌ Modal harus 10,000 - 5,000,000")
        except Exception:
            print("❌ Input tidak valid")

    if engine_choice == '1':
        config = SwingConfig(modal)
        engine_name = "SWING ENGINE"
        EngineClass = SwingEngine
        timeframe = '1d'
    elif engine_choice == '2':
        config = IntradayConfig(modal)
        engine_name = "INTRADAY LIQUID ENGINE"
        EngineClass = IntradayLiquidEngine
        timeframe = '1h'
    elif engine_choice == '3':
        config = GorenganConfig(modal)
        engine_name = "INTRADAY GORENGAN ENGINE"
        EngineClass = IntradayGorenganEngine
        timeframe = '1h'
    else:
        config = InvestasiConfig(modal)
        engine_name = "INVESTASI ENGINE"
        EngineClass = InvestasiEngine
        timeframe = '1d'

    modal_adapter = ModalAdapter(modal)
    modal_adapter.print_info()

    print("\n🚀 Initializing Phase 1 Components...")

    risk_manager = RiskManager(
        modal=modal,
        risk_per_trade_pct=0.5,
        max_risk_portfolio_pct=3.0,
        max_lot_per_position=5
    )
    print(f"   ✅ Risk Manager: Rp {risk_manager.risk_per_trade_rp:,.0f} risk/trade (STABIL 0.5%)")

    fee_config = RealisticFeeConfig(liquidity='medium')
    print(f"   ✅ Fee Config: min fee Rp {fee_config.MIN_FEE_PER_TRANSACTION:,}")

    global_fetcher = GlobalIndicesFetcher()
    global_fetcher.fetch_all()
    global_fetcher.print_detailed_report()

    backtester = AccurateBacktester(initial_capital=modal)
    backtester.set_fee_config(fee_config)
    print(f"   ✅ Backtester: ready")

    delay_simulator = EntryDelaySimulator(max_delay=2)
    print(f"   ✅ Entry Delay Simulator: max delay {delay_simulator.max_delay} days")

    portfolio_risk = PortfolioRiskCalculator(lookback_days=60)
    print(f"   ✅ Portfolio Risk Calculator: {portfolio_risk.lookback_days} days lookback")

    engine = EngineClass(config, global_fetcher)
    engine.set_risk_manager(risk_manager)

    print("\n" + "="*60)
    print(f"📊 {engine_name} - READY")
    print("="*60)
    print(f"Modal: Rp {modal:,}")
    print(f"Risk per trade: Rp {risk_manager.risk_per_trade_rp:,.0f} (STABIL 0.5%)")
    print(f"Max portfolio risk: Rp {risk_manager.max_risk_portfolio_rp:,.0f} (STABIL 3.0%)")
    print(f"Min fee: Rp {fee_config.MIN_FEE_PER_TRANSACTION:,}")
    print(f"Universe: {len(symbols)} saham valid dari warehouse")

    print(f"\n📥 Pilih jumlah saham yang akan dianalisis:")
    print(f"   1. Semua ({len(symbols)} saham) - Lebih lama tapi komprehensif")
    print(f"   2. Sample 100 saham - Sedang")
    print(f"   3. Sample 50 saham - Cepat")
    print(f"   4. Sample 20 saham - Sangat cepat")

    while True:
        sample_choice = input("Pilihan (1/2/3/4): ").strip()
        if sample_choice == '1':
            test_symbols = symbols
            break
        elif sample_choice == '2':
            test_symbols = symbols[:100]
            break
        elif sample_choice == '3':
            test_symbols = symbols[:50]
            break
        elif sample_choice == '4':
            test_symbols = symbols[:20]
            break
        else:
            print("❌ Pilih 1, 2, 3, atau 4")

    print(f"\n📥 Mengambil data dari warehouse untuk {len(test_symbols)} saham...")

    stocks_data = {}
    for i, symbol in enumerate(test_symbols):
        df = warehouse.get_data(symbol)
        if df is not None:
            stocks_data[symbol] = df

        if (i + 1) % 50 == 0:
            print(f"   Progress: {i+1}/{len(test_symbols)} - {len(stocks_data)} dimuat")

    print(f"\n✅ Berhasil memuat {len(stocks_data)} saham dari warehouse")

    if stocks_data:
        print(f"\n📊 Menganalisis {len(stocks_data)} saham...")
        signals = []

        for symbol, df in stocks_data.items():
            signal = engine.get_signal(symbol, df)
            if signal:
                signals.append(signal)

            if len(signals) % 10 == 0 and len(signals) > 0:
                print(f"   Ditemukan {len(signals)} sinyal sejauh ini...")

        print(f"✅ Ditemukan {len(signals)} sinyal")

        if signals:
            print("\n" + "="*100)
            print("🌍 RINGKASAN PASAR")
            print("="*100)
            market_data = []
            for name in GLOBAL_INDICES.keys():
                mom = global_fetcher.get_momentum(name)
                trend = global_fetcher.get_trend(name)
                price_str = global_fetcher.get_price_str(name)
                market_data.append([name, price_str, f"{mom:+.2f}%", trend])
            print(tabulate(market_data, headers=["Indeks", "Harga", "Momentum", "Trend"], tablefmt="grid"))

            print("\n" + "="*100)
            print(f"📊 {engine_name} - REKOMENDASI ({len(signals)} sinyal dari {len(stocks_data)} saham)")
            print("="*100)

            if engine_choice == '4':
                display_data = []
                for s in signals[:20]:
                    display_data.append([
                        s['Symbol'],
                        s['Sector'],
                        f"Rp {s['Price']:,}",
                        f"{s['To_MA50']}",
                        f"Rp {s['MA50']:,}",
                        f"Rp {s['MA200']:,}",
                        f"Rp {s['Stop_Loss']:,}",
                        s['Take_Profit'],
                        f"{s['Risk_Pct']}%",
                        f"{s['Lot']} lot",
                        f"Rp {s['Cost']:,}"
                    ])

                headers = [
                    "Kode", "Sektor", "Harga", "To MA50", "MA50", "MA200",
                    "Stop Loss", "Target", "Risk%", "Lot", "Biaya"
                ]
            else:
                display_data = []
                for s in signals[:20]:
                    display_data.append([
                        s['Symbol'],
                        s['Sector'],
                        f"Rp {s['Price']:,}",
                        s.get('RSI', '-'),
                        s.get('Volume', '-'),
                        f"{s['R/R']}",
                        f"{s['EV_Pct']}%",
                        f"{s['Score']}",
                        f"Rp {s['Stop_Loss']:,}",
                        f"Rp {s['Take_Profit']:,}",
                        f"{s['Risk_Pct']}%",
                        f"{s['Lot']} lot",
                        f"Rp {s['Cost']:,}"
                    ])

                headers = [
                    "Kode", "Sektor", "Harga", "RSI", "Vol", "R/R",
                    "EV%", "Skor", "SL", "TP", "Risk%", "Lot", "Biaya"
                ]

            print(tabulate(display_data, headers=headers, tablefmt='grid', stralign='left', numalign='center'))

            if len(signals) > 20:
                print(f"\n... dan {len(signals) - 20} sinyal lainnya (total {len(signals)})")

            top_signals = sorted(signals, key=lambda x: x.get('Score', 0), reverse=True)[:5]
            total_risk = sum([s['Risk_Amount'] for s in top_signals])
            total_risk_pct = (total_risk / modal) * 100
            print(f"\n📊 Total Portfolio Risk (5 sinyal terbaik): Rp {total_risk:,} ({total_risk_pct:.1f}% dari modal)")
            print(f"   (Maksimum risk: {risk_manager.max_risk_portfolio_pct}% - STABIL)")

            # ===== PANDUAN PORTFOLIO 3 TERBAIK =====
            print_portfolio_guide(signals, modal, risk_manager)

            # ===== EXPORT KE GOOGLE SHEETS =====
            export_choice = input(f"\n📊 Export {engine_name} signals ke Google Sheets? (y/n): ").strip().lower()
            if export_choice == 'y':
                sheets_exporter.export_signals(signals, engine_name, modal)

        else:
            print("\n❌ Tidak ada sinyal hari ini")

            if modal < 100000:
                print("\n💡 SARAN UNTUK MODAL KECIL:")
                print("   - Filter sudah dilonggarkan (harga rendah, spread besar)")
                print("   - Coba engine Intraday Gorengan untuk lebih banyak sinyal")
    else:
        print("\n❌ Tidak ada data yang berhasil dimuat dari warehouse")

    print("\n" + "="*60)
    print("✅ PHASE 1 COMPLETE")
    print("   Risk Management STABIL: 0.5% per trade, 3% portfolio")
    print("="*60)

# =============================================================================
# 28. PHASE 2 - MAIN PROGRAM
# =============================================================================

def run_phase2():
    """Main program Phase 2 - Validation dengan Monte Carlo FIXED"""

    print("\n" + "="*80)
    print("📊 IDX STOCK SCANNER - PHASE 2 VALIDATION")
    print("   Monte Carlo FIXED | Walk-Forward | Sensitivity")
    print("="*80)

    warehouse = DataWarehouse(warehouse_dir='data_warehouse', min_days=400)
    symbols = warehouse.get_all_valid_symbols()

    if not symbols:
        print("\n⚠️  Data warehouse kosong!")
        print("   Jalankan opsi 3 (Inisialisasi Data Warehouse) terlebih dahulu.")
        return

    print(f"\n📥 Loading data dari warehouse ({len(symbols)} saham tersedia)...")

    print(f"\n📊 Pilih jumlah saham untuk validasi:")
    print(f"   1. Semua ({len(symbols)} saham)")
    print(f"   2. Sample 200 saham")
    print(f"   3. Sample 100 saham")

    while True:
        sample_choice = input("Pilihan (1/2/3): ").strip()
        if sample_choice == '1':
            test_symbols = symbols
            break
        elif sample_choice == '2':
            test_symbols = symbols[:200]
            break
        elif sample_choice == '3':
            test_symbols = symbols[:100]
            break
        else:
            print("❌ Pilih 1, 2, atau 3")

    data_dict = {}
    for i, symbol in enumerate(test_symbols):
        df = warehouse.get_data(symbol)
        if df is not None:
            data_dict[symbol] = df

        if (i+1) % 50 == 0:
            print(f"   Loaded {i+1}/{len(test_symbols)} - {len(data_dict)} valid")

    print(f"\n✅ Loaded {len(data_dict)} stocks dengan data lengkap dari warehouse")

    if not data_dict:
        print("❌ No data loaded. Cannot proceed.")
        return

    print("\nPilih engine untuk divalidasi:")
    print("1. Swing Engine")
    print("2. Intraday Liquid")
    print("3. Intraday Gorengan")
    print("4. Investasi Engine")

    while True:
        choice = input("Pilihan (1/2/3/4): ").strip()
        if choice in ['1', '2', '3', '4']:
            break
        print("❌ Pilih 1-4")

    if choice == '1':
        engine_class = SwingEngine
        config = SwingConfig(modal=100000000)
        engine_name = "SWING ENGINE"
    elif choice == '2':
        engine_class = IntradayLiquidEngine
        config = IntradayConfig(modal=100000000)
        engine_name = "INTRADAY LIQUID ENGINE"
    elif choice == '3':
        engine_class = IntradayGorenganEngine
        config = GorenganConfig(modal=100000000)
        engine_name = "INTRADAY GORENGAN ENGINE"
    else:
        engine_class = InvestasiEngine
        config = InvestasiConfig(modal=100000000)
        engine_name = "INVESTASI ENGINE"

    print(f"\n🔧 Validating: {engine_name} dengan {len(data_dict)} saham")

    suite = ValidationSuite(engine_class, config, data_dict)
    results = suite.run_all()

    print("\n" + "="*80)
    print("✅ PHASE 2 VALIDATION COMPLETE")
    print("   File output: monte_carlo_bs*_fixed.png")
    print("="*80)

# =============================================================================
# 29. MAIN MENU
# =============================================================================

def main():
    """Main menu untuk memilih Phase"""

    print("\n" + "="*70)
    print("🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE")
    print("   FULL SYSTEM DENGAN DATA WAREHOUSE")
    print("   Modal-Adaptive Filters | Risk Stabil 0.5%")
    print("   Export Google Sheets | Panduan Portfolio 3 Terbaik")
    print("="*70)

    # Inisialisasi Google Sheets Exporter (URL akan tampil di awal)
    sheets_exporter = GoogleSheetsExporter()
    sheets_exporter.ensure_spreadsheet_exists()

    print("\nPilih Mode:")
    print("1. Phase 1 - Trading Scanner (Live Signals) - Dengan Export & Panduan")
    print("2. Phase 2 - Validation Suite (Monte Carlo FIXED)")
    print("3. Inisialisasi Data Warehouse (Download historis lengkap)")
    print("4. Exit")

    while True:
        choice = input("\nPilihan (1/2/3/4): ").strip()
        if choice == '1':
            run_phase1(sheets_exporter)
            break
        elif choice == '2':
            run_phase2()
            break
        elif choice == '3':
            initialize_data_warehouse()
            break
        elif choice == '4':
            print("\n✅ Exiting...")
            break
        else:
            print("❌ Pilih 1, 2, 3, atau 4")

if __name__ == "__main__":
    main()

✅ Dependencies imported

🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE
   FULL SYSTEM DENGAN DATA WAREHOUSE
   Modal-Adaptive Filters | Risk Stabil 0.5%
   Export Google Sheets | Panduan Portfolio 3 Terbaik
✅ Berhasil konek ke Google Sheets
✅ Membuka spreadsheet: IDX_Scanner_All_Engines

📊 Google Sheets URL: https://docs.google.com/spreadsheets/d/10ITzuSx_6512KrfSmkqaRPaAsFbPi6Z_NO-6e6u9BKg

Pilih Mode:
1. Phase 1 - Trading Scanner (Live Signals) - Dengan Export & Panduan
2. Phase 2 - Validation Suite (Monte Carlo FIXED)
3. Inisialisasi Data Warehouse (Download historis lengkap)
4. Exit

Pilihan (1/2/3/4): 1

🏦 IDX STOCK SCANNER - QUADRUPLE ENGINE
   Modal-Adaptive Filters | Risk Stabil 0.5% per trade
   Export: Google Sheets | Panduan Portfolio 3 Terbaik

📊 Data warehouse siap: 640 saham dengan data >= 400 hari

Pilih engine trading:
1. Swing Engine (Mingguan - Mean Reversion)
2. Intraday Liquid (1 jam - Momentum)
3. Intraday Gorengan (1 jam - Early Momentum)
4. Investasi Engine (Quality + T